# QEC The Basics of Classical and Quantum Error Correction

$
\renewcommand{\ket}[1]{|{#1}\rangle}
\renewcommand{\bra}[1]{\langle{#1}|}
$

Execute the cells below to load all the necessary packages for this lab.

In [ ]:
!pip install cudaq
!pip install cudaq_qec
!sudo apt-get update && sudo apt-get install -y gfortran
!pip install cudaq-solvers 

In [ ]:
# install `qutip` and `ipywidgets` in the current Python kernel. Skip this if they are already installed.
# `matplotlib` is required for all visualization tasks.
# Make sure to restart your kernel if you execute this!
# In a Jupyter notebook, go to the menu bar > Kernel > Restart Kernel.
# In VSCode, click on the Restart button in the Jupyter toolbar.

# The '\' before the '>' operator is so that the shell does not misunderstand
# the '>' qualifier for the bash pipe operation.

import sys

try:
    import matplotlib.pyplot as plt
    import qutip
    import ipywidgets as widgets
    import matplotlib_venn

except ImportError:
    print("Tools not found, installing. Please restart your kernel after this is done.")
    !{sys.executable} -m pip install qutip\>=5 matplotlib\>=3.5 matplotlib_venn
    !{sys.executable} -m pip install ipywidgets
    print("\nNew libraries have been installed. Please restart your kernel!")

In [ ]:
import cudaq
from cudaq import spin
from cudaq.qis import *
import numpy as np
import random
import matplotlib.pyplot as plt


from typing import List
import ipywidgets as widgets
from ipywidgets import interact, Output, VBox, HBox
from IPython.display import display

## 1.1 The Basics of Error Correction

Classical [Error correction](https://en.wikipedia.org/wiki/Error_correction_code) (EC) is the practice of redundantly encoding data using additional bits such that if some of the data bits are corrupted (flipped from a 0 to a 1 or vice versa) by external noise, the error can be fixed and the integrity of the original data preserved. There are many potential ways to accomplish this and any specific procedure is referred to as an EC code.

It is thanks to EC that you can still watch a DVD that has many scratches, enjoy a clear telephone conversation despite the signal becoming noisy after transmission over long distances, or scan a QR code at a restaurant that is partially obscured. Try it yourself. Pull up a QR code and try covering different parts with your hand and notice that your phone can still interpret the link despite obfuscation of some parts of the grid. EC has its limits, but in many cases can essentially eliminate any negative impacts of noise.

There are five aspects to any general EC procedure:

1. All EC procedures assume there is some initial information that needs to be preserved or transmitted. The sender and recipient need to interact with the same information even if it is stored in different ways throughout an EC procedure.

2.  An EC procedure first **encodes** the information across $n$ data bits such that $n$ is larger than the minimum number of bits necessary to store the information. For example, the repitition code that we'll cover in the next section uses 3 bits to encode some binary information stored on a single bit by simply repeating the information stored on the single bit three times.

3. The encoding procedure produces **logical codewords** which are the redundant encodings that map to the logical states.  For example, logical 1 could be defined with the codeword 111.

4. A logical codeword then proceeds through a **noisy channel**.  A noisy channel has some probability of randomly corrupting (flipping) any of the data bits.

5.  The recipient then receives the encoded data and needs to decode it to determine if an error occurred and how they might fix it. Another way to say this, is that the **decoder** takes the message the recipient receives and determines which logical codeword is "closest" to it. If the decoder produces the correct logical codeword, the EC procedure worked. If not, a **logical error** occurred and the recipient incorrectly interprets the message - the worst case scenario.

No EC procedure is perfect, and is usually benchmarked against **[Shannon's limit](https://en.wikipedia.org/wiki/Noisy-channel_coding_theorem)** which is the theoretical limit of the rate of noise free data transfer that can occur through a channel with a given bandwidth and noise level.

In practice, and EC code need to be just "good enough" to ensure that errors do not usually impede the target application. Generally speaking, the more redundancy (extra bits) used in the encoding process, the better the EC will be, but this also comes at the cost of more memory and time to perform more sophisticated encoding and decoding. Thus, there is always a tension between the resources available for error correction and the logical error rate of the procedure necessary for a given application.

## 1.2 The Repetition Code

The most basic EC code is called the repetition code.

Consider encoding the information in a single bit (0 or 1). The repetition code simply adds more bits which are in the same state. So a 3-bit repetition code encodes the logical 0 state ($0_L$) as 000 and the logical 1 state ($1_L$) as 111, making 000 and 111 the logical codewords.

Now, assume $0_L$ is transmitted through a noisy channel such that each data qubit has a probability p=0.1 of flipping erroneously. This means there are eight possible states the encoded message could be in after proceeding through the noisy channel. The states can be sorted into two groups. The space of all logical codewords (000 and 111) is called the **codespace**:

| Codespace (as bitstrings)    | Codespace (as logical states)|
| ----------- | ----------- |
| 000 | $0_L$ |
| 111 | $1_L$ |


while the rest of all the potentially received messages belong to the **error space** as shown in the table below.

The job of the decoder is to map encodings in the error space back to a logical codeword in the codespace.  This is usually done through the help of a calculated property that describes the state called a **syndrome**.  In this case, the syndrome is a majority count of the bits. So, the syndrome of the 101 state would be 2 (the count of how many 1's) and corresponds to logical 1 and the syndrome of 001 would be 1.  Additionally, the syndrome of 111 is 3 and the syndrome of 000 os 0.  So we can apply the decoding rule that a message with a syndrome of 2 or 3 would be decoded as 111, and message with a syndrome of 0 or 1 would be decoded as 000.

| Error Space   | Closest logical state to the received message | Error, if only one error occurred in transmission | Syndrome |
| ----------- | ----------- | ----------- | ----------- |
| 001 | 000 | right most bit | 1|
| 010 | 000 | middle bit | 1|
| 100 | 000 | left most bit | 1 |
| 110 | 111 | right most bit | 2|
| 101 | 111 | middle bit |  2 |
| 011 | 111 |left most bit | 2 |

Any single bit error can be correctly identified and corrected, but two or more bit flips will result in a logical error.  There is no way to know for certain if 110, for example, is a single error from the $1_L$ codeword or two bit errors from the $0_L$ codeword. However, the repetition code still greatly reduces the probability of a logical error compared to no encoding. To be very conservative you could discard results where any error was detected and resend the message.  In this error detection (not correction) case,  a logical error will only occur in the highly unlikely case of three bit flip errors, where 000 was transmitted but 111 was received.  This approach requires additional data transfers to compensate for the cases where errors occurred. Error correction can eliminate the need for extra data transfers at the expense of possibly mistaking a 2-bit error for a 1-bit error.  This hints at an important property of error correction codes: distance.

Codes are characterized with the so called [n,k,d] notation where n is the number of data bits used to encode k logical bits and d is the **distance**. The distance is the number of errors that must occur for one logical codeword to become the next closest codeword. In our example of the three bit repetition code $n=3$, $k = 1$, and $d=3$, so we refer to this as a [3,1,3] code. The number of errors a code can correct $t$ is a function of code distance and can be calculated as $t = \text{floor}[(d-1)/2]$

The table below shows the likelihood of the four possible scenarios below. Notice the three bit repetition code with majority count will transmit the message with 0.972 probability of success, a significant improvement over the original probability of 0.9.  The **logical error rate** is equal to $1-p$, where $p$ is the probability of success.  In the case of the 3-bit repetition code, the logical error rate is 0.028.

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-6-images/errortable.png?raw=1" alt="Drawing" style="width: 600px;"/>





In [ ]:
### Exercise  1 - The repetition code:

You now know enough to code up the repetition code. The exercise below will require you to generalize the repetition code so it will work with $n$ bits. Fill in the #TODO sections and then observe the plots that are generated. What conclusions can you draw from the code performance using more bits?  What do you notice about the logical error rate relative to the physical error rate


In [ ]:
def encode(bit, n):
    """Function that encodes a single bit rendundantly n times

    Parameters
    ----------
    bit: int
        Input bit (1 or 0)
    n : int
        repetitions to use for encoding

    Returns
    -------
    str
        string of length n redundantly encoding bit
    """
    #TODO



def decode(bits):
    """Function that decodes a message using majority voting to determine the closest codeword

    Parameters
    ----------
    bits: str
        bitstring corresponding to message that has passed through noisy channel

    Returns
    -------
    int
        1 or 0 corresponding to decoded codeword
    """
    #TODO



def transmit(bits, p_error):
    """Function that receives a codeword, and randomly flips each bit with probability p_error to emulate transmission through noisy channel

    Parameters
    ----------
    bits: str
        bitstring corresponding to an encoded message without noise
    p_error: float
        probability that a bit will flip through transmission

    Returns
    -------
    int
        1 or 0 corresponding to decoded codeword
    """
    #TODO


def simulate_logical_error_rate(n, p_error, trials):
    """Function to determine the logical error rate of an n-bit repetition code over specified number of trials.

    Parameters
    ----------
    n: int
        specifies n-bit repetition code to use
    p_error: float
        probability that a bit will flip through transmission
    trials: int
        number of trials used to determine logical error rate

    Returns
    -------
    float
        The logical error rate `n_errors/trials`
    """
#TODO



def plot_logical_vs_physical_error_rate(n, trials):
    """Function to plot logical vs physical error rate for fixed n and number of trials.

    Parameters
    ----------
    n: int
        specifies n-bit repetition code to use
    trials: int
        number of trials used to determine logical error rate
    """

    #TODO

    plt.figure(figsize=(10, 6))
    plt.plot(p_values, logical_error_rates, marker='o')
    plt.title('Logical Error Rate vs Physical Error Rate')
    plt.xlabel('Physical Error Rate')
    plt.ylabel('Logical Error Rate')
    plt.grid(True)
    plt.show()

# Plot 2: Logical Error Rate vs n
def plot_logical_vs_repetitions(p_error, max_n, trials):
    """Function to plot logical error rate vs bits used for redundant encoding

    Parameters
    ----------
    max_n: int
        specifies the maximum n-bit repetition code to use
    p_error: float
        probability that a bit will flip through transmission
    trials: int
        number of trials used to determine logical error rate
    """

    #TODO
    plt.figure(figsize=(10, 6))
    plt.plot(n_values, logical_error_rates, marker='o')
    plt.title('Logical Error Rate vs n')
    plt.xlabel('n')
    plt.ylabel('Logical Error Rate')
    plt.grid(True)
    plt.show()

# Example Usage
n = 3         # Number of repetitions for the first plot
p_error = 0.1       # Physical error rate for the second plot
trials = 10000

# Generate the plots
plot_logical_vs_physical_error_rate(n, trials)

max_n = 20
plot_logical_vs_repetitions(p_error, max_n, trials)

## 1.3 More Efficient EC Codes (The Hamming Code)

There are many clever ways to improve the efficiency of EC codes. One common way is to make use of a concept called **parity checks**. Parity checks provide a clever way to index where errors occur, without a brute force statistical approach like the repetition code.

[The Hamming code](https://en.wikipedia.org/wiki/Hamming_code) is a great example of a parity check code.  The [7,4,3] Hamming code consists of four data bits ($d_1, d_2, d_3, d_4$) and three additional parity check bits ($p_1, p_2, p_3$). This example will only consider single bit errors as this is another distance 3 code and can only correct single bit errors.

This is accomplished by each parity bit encoding a parity, or the mod2 sum of a subset of the data bits.  The [Venn diagram](https://en.wikipedia.org/wiki/Hamming_code) below depicts the encoding.  In this example, $p_1$ encodes the parity of $d_1$, $d_2$, and $d_4$.  If our data bits ($d_1d_2d_3d_4$) were 0110, then $p_1$ would be calculated to be 1.

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-6-images/Hamming(7,4).svg?raw=1" alt="Drawing" style="width: 300px;"/>

Either using the static Venn diagram above or the interactive one generated by executing the cell below,
reason through the following example:

> If you wanted to send the message 0110 (here $d_1 = 0$, $d_2 = 1$, $d_3 = 1$, and $d_4 = 0$), appending the three parity bits to the end of the original bitstring would produce the logical codeword: 0110110 (where $p_1 = 1$, $p_2 = 1$, and $p_3 = 0$).  Note, this is a slight deviation from the traditional placement of the bits in the Hamming code done for simplicity.
>
>Errors could occur on any of the data or parity bits. Assume an error occurs on $d_2$ and the recipient receives 0010110. To produce the syndrome, the recipient can take the received data bits, 0010, and compute the expected parity. This is then compared to the parity that was sent, 110. The parity bits that disagree flag an error.
>
>In this case, the received message has parity bits 011 which disagrees with 110. Here, $p_1$ and $p_3$ are flagged.  This syndrome can only correspond to an error on $d_2$ based on the Venn diagram.  



In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn3
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from ipywidgets import VBox, HBox

# Function to calculate parity bits
def calculate_parity_bits(data_bits):
    d1, d2, d3, d4 = data_bits
    p1 = d1 ^ d2 ^ d4
    p2 = d1 ^ d3 ^ d4
    p3 = d2 ^ d3 ^ d4
    return [p1, p2, p3]

# Function to update the Venn diagram labels based on data bits
def update_venn_labels(data_bits):
    # Clear the previous output
    clear_output(wait=True)

    # Clear the computed parity bit outputs
    output_p1.clear_output()
    output_p2.clear_output()
    output_p3.clear_output()

    # Display the widgets again
    display(VBox([title, data_bits_widget, HBox([button_p1, button_p2, button_p3], layout=widgets.Layout(justify_content='space-between')), HBox([output_p1, output_p2, output_p3])]))

    # Create the Venn diagram
    plt.figure(figsize=(8, 8))
    venn = venn3(subsets=(1, 1, 1, 1, 1, 1, 1), set_labels=('p1', 'p2', 'p3'))

    # Set colors for the circles using NVIDIA color palette
    venn.get_patch_by_id('100').set_color('#76B900')  # Green
    venn.get_patch_by_id('010').set_color('#7A1FA2')  # Purple
    venn.get_patch_by_id('001').set_color('#F9A825')  # Yellow

    # Set colors for the intersections
    venn.get_patch_by_id('110').set_color('#A3A3A3')  # Light Gray
    venn.get_patch_by_id('101').set_color('#A3A3A3')  # Light Gray
    venn.get_patch_by_id('011').set_color('#A3A3A3')  # Light Gray
    venn.get_patch_by_id('111').set_color('#A3A3A3')  # Light Gray

    # Set transparency for the circles
    venn.get_patch_by_id('100').set_alpha(0.5)
    venn.get_patch_by_id('010').set_alpha(0.5)
    venn.get_patch_by_id('001').set_alpha(0.5)

    # Label the intersections with data bits
    venn.get_label_by_id('100').set_text(f'')
    venn.get_label_by_id('010').set_text(f'')
    venn.get_label_by_id('001').set_text(f'')
    venn.get_label_by_id('110').set_text(f'd1={data_bits[0]}')
    venn.get_label_by_id('101').set_text(f'd2={data_bits[1]}')
    venn.get_label_by_id('011').set_text(f'd3={data_bits[2]}')
    venn.get_label_by_id('111').set_text(f'd4={data_bits[3]}')

    plt.show()

# Function to update the Venn diagram and display the messages
def update_venn(data_bits, parity_bit):
    parity_bits = calculate_parity_bits(data_bits)

    # Clear the previous output
    clear_output(wait=True)

    # Display the widgets again
    display(VBox([title, data_bits_widget, HBox([button_p1, button_p2, button_p3], layout=widgets.Layout(justify_content='space-between')), HBox([output_p1, output_p2, output_p3])]))

    # Create the Venn diagram
    plt.figure(figsize=(8, 8))
    venn = venn3(subsets=(1, 1, 1, 1, 1, 1, 1), set_labels=('p1', 'p2', 'p3'))

    # Set colors for the circles using NVIDIA color palette
    venn.get_patch_by_id('100').set_color('#76B900')  # Green
    venn.get_patch_by_id('010').set_color('#7A1FA2')  # Purple
    venn.get_patch_by_id('001').set_color('#F9A825')  # Yellow

    # Set colors for the intersections
    venn.get_patch_by_id('110').set_color('#A3A3A3')  # Light Gray
    venn.get_patch_by_id('101').set_color('#A3A3A3')  # Light Gray
    venn.get_patch_by_id('011').set_color('#A3A3A3')  # Light Gray
    venn.get_patch_by_id('111').set_color('#A3A3A3')  # Light Gray


    # Set transparency for the circles
    venn.get_patch_by_id('100').set_alpha(0.5)
    venn.get_patch_by_id('010').set_alpha(0.5)
    venn.get_patch_by_id('001').set_alpha(0.5)

    # Label the intersections with data bits
    venn.get_label_by_id('100').set_text(f'')
    venn.get_label_by_id('010').set_text(f'')
    venn.get_label_by_id('001').set_text(f'')
    venn.get_label_by_id('110').set_text(f'd1={data_bits[0]}')
    venn.get_label_by_id('101').set_text(f'd2={data_bits[1]}')
    venn.get_label_by_id('011').set_text(f'd3={data_bits[2]}')
    venn.get_label_by_id('111').set_text(f'd4={data_bits[3]}')

    # Highlight the selected parity bit and relevant data bits
    if parity_bit == 'p1':
        venn.get_patch_by_id('100').set_edgecolor('black')
        venn.get_patch_by_id('100').set_linewidth(5)
        venn.get_patch_by_id('110').set_color('#76B900')  # Green
        venn.get_patch_by_id('101').set_color('#76B900')  # Green
        venn.get_patch_by_id('111').set_color('#76B900')  # Green
        venn.get_patch_by_id('110').set_edgecolor('black')
        venn.get_patch_by_id('110').set_linewidth(5)
        venn.get_patch_by_id('101').set_edgecolor('black')
        venn.get_patch_by_id('101').set_linewidth(5)
        venn.get_patch_by_id('111').set_edgecolor('black')
        venn.get_patch_by_id('111').set_linewidth(5)
        output_p1.clear_output()
        with output_p1:
            display(HTML(f"<b>p1 = d1 + d2 + d4 (mod 2)=</b> {data_bits[0]} + {data_bits[1]} + {data_bits[3]} (mod 2) = {parity_bits[0]}"))
    elif parity_bit == 'p2':
        venn.get_patch_by_id('010').set_edgecolor('black')
        venn.get_patch_by_id('010').set_linewidth(5)
        venn.get_patch_by_id('110').set_color('#7A1FA2')  # Purple
        venn.get_patch_by_id('011').set_color('#7A1FA2')  # Purple
        venn.get_patch_by_id('111').set_color('#7A1FA2')  # Purple
        venn.get_patch_by_id('110').set_edgecolor('black')
        venn.get_patch_by_id('110').set_linewidth(5)
        venn.get_patch_by_id('011').set_edgecolor('black')
        venn.get_patch_by_id('011').set_linewidth(5)
        venn.get_patch_by_id('111').set_edgecolor('black')
        venn.get_patch_by_id('111').set_linewidth(5)
        output_p2.clear_output()
        with output_p2:
            display(HTML(f"<b>p2 = d1 + d3 + d4 (mod 2)=</b> {data_bits[0]} + {data_bits[2]} + {data_bits[3]} (mod 2) = {parity_bits[1]}"))
    elif parity_bit == 'p3':
        venn.get_patch_by_id('001').set_edgecolor('black')
        venn.get_patch_by_id('001').set_linewidth(5)
        venn.get_patch_by_id('101').set_color('#F9A825')  # Yellow
        venn.get_patch_by_id('011').set_color('#F9A825')  # Yellow
        venn.get_patch_by_id('111').set_color('#F9A825')  # Yellow
        venn.get_patch_by_id('101').set_edgecolor('black')
        venn.get_patch_by_id('101').set_linewidth(5)
        venn.get_patch_by_id('011').set_edgecolor('black')
        venn.get_patch_by_id('011').set_linewidth(5)
        venn.get_patch_by_id('111').set_edgecolor('black')
        venn.get_patch_by_id('111').set_linewidth(5)
        output_p3.clear_output()
        with output_p3:
            display(HTML(f"<b>p3 = d2 + d3 + d4 (mod 2)=</b> {data_bits[1]} + {data_bits[2]} + {data_bits[3]} (mod 2) = {parity_bits[2]}"))

    plt.show()

# Create a title widget
title = widgets.Label(value="Hamming Code Visualization: Computing parity bits (p1, p2, p3)")


# Create widgets for user input
data_bits_widget = widgets.Dropdown(
    options=['0000', '0001', '0010', '0011', '0100', '0101', '0110', '0111', '1000', '1001', '1010', '1011', '1100', '1101', '1110', '1111'],
    value='1001',
    description='Data Bits (d1, d2, d3, d4):', style={'description_width': 'initial'}
)

# Create buttons for parity bits
button_p1 = widgets.Button(description='Compute p1', layout=widgets.Layout(width='150px'), style=widgets.ButtonStyle(button_color='#BBE07F'))  # Green
button_p2 = widgets.Button(description='Compute p2', layout=widgets.Layout(width='150px'), style=widgets.ButtonStyle(button_color='#BD8FD1'))  # Purple
button_p3 = widgets.Button(description='Compute p3', layout=widgets.Layout(width='150px'), style=widgets.ButtonStyle(button_color='#FCD492'))  # Yellow

# Create output areas for parity bit results
output_p1 = widgets.Output(layout=widgets.Layout(width='300px'))
output_p2 = widgets.Output(layout=widgets.Layout(width='300px'))
output_p3 = widgets.Output(layout=widgets.Layout(width='300px'))

# Define the button click events
def on_button_p1_click(b):
    data_bits_list = [int(bit) for bit in data_bits_widget.value]
    update_venn(data_bits_list, 'p1')

def on_button_p2_click(b):
    data_bits_list = [int(bit) for bit in data_bits_widget.value]
    update_venn(data_bits_list, 'p2')

def on_button_p3_click(b):
    data_bits_list = [int(bit) for bit in data_bits_widget.value]
    update_venn(data_bits_list, 'p3')

button_p1.on_click(on_button_p1_click)
button_p2.on_click(on_button_p2_click)
button_p3.on_click(on_button_p3_click)

# Define the dropdown change event
def on_data_bits_change(change):
    data_bits_list = [int(bit) for bit in change['new']]
    update_venn_labels(data_bits_list)

data_bits_widget.observe(on_data_bits_change, names='value')

# Display the widgets
display(VBox([title, data_bits_widget, HBox([button_p1, button_p2, button_p3], layout=widgets.Layout(justify_content='space-between')), HBox([output_p1, output_p2, output_p3])]))

# Initial update of the Venn diagram labels
update_venn_labels([int(bit) for bit in data_bits_widget.value])


The Hamming code takes advantage of the fact that the parity bits can encode up to $2^3 = 8$ syndromes, more than enough to consider the seven possible single bit flip errors that could occur. This means, 4 bits can be encoded with 7 bits which is an improvement over the $n$ to 1 encoding of the repetition code.

**Checkpoint:**  Suppose you sent the logical code word 0110110, but the recipient received the message 0110100.  We'll assume that at most only one error occurred.  Would the recipient be able to identify if there was an error?  If so, could they locate the error?  
Hint: errors could occur on any of the data or the parity bits.

In practice, a large message can be broken into blocks with each block is encoded using the Hamming code.  The code scales much better than the repetition code.  A Hamming code can be produced for any integer $r$ greater than 1, such that the code is characterized as $[2^r-1,2^r -r -1,3]$.  So for a message of size $2^5 -5 -1 = 26$, the Hamming code would require only $2^5-1 =31$ bits while a three bit repetition code would require $26*3 = 78$ bits.

### Exercise  2 - The matrix form of the Hamming code:
    
The Hamming code is commonly constructed with special matrices so a few simple linear algebra operations can encode and decode messages. The next two cells will have you define these matrices and see if you can reproduce the example above.  


First, define the generator matrix $G$ such that a dot product between the message and $G$  mod2 performs the valid encoding. Hint: G should be a 4x7 matrix.

In [ ]:
message = np.array([0, 1, 1, 0])

# The G matrix should properly encode the message when the following calculation is performed
G = np.array([
#FILL IN G.
])

encoded = np.dot(message, G) % 2
print(encoded)

Now, define the parity check matrix $H$ such that $Hv \mod 2$ produces a syndrome, where $v$ is the received message vector.

In [ ]:
received = np.array([0, 0, 1, 0,1,1,0])
print(received)

# Define the parity check matrix H which takes a message and determines the syndrome.
H = np.array([
#FILL IN H
])

decoded = np.dot(H, recieved) % 2

# Should print [0 1 1]
print(decoded)

## 1.4 What Makes QEC so Hard?

QEC shares the same goal as classical EC, but comes with a number of unique challenges thanks to the properties of quantum mechanics.  This section will list the primary differences, and the following section will explain how these challenges can be addressed.

1.  Continuous Errors - Classical errors are always discrete bit flips. Quantum errors are continuous and can manifest in an infinite number of ways, potentially shifting a qubit's state to any point on the Bloch sphere. For instance, the figure below illustrates many possible errors that affect a qubit starting in the $\ket{0}$ state. Errors can perturb states incoherently (from environmental effects) or coherently from slight hardware imperfections. This invites the question, "Does QEC require an infinite amount of resources to correct errors?"
   
<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-6-images/c_error.png?raw=1" alt="Drawing" style="width: 300px;"/>

2. No Cloning - Quantum states cannot be copied. That is to say that the following expression holds:$~\nexists U \text{ such that } U(\ket{\psi} \otimes \ket{\rho}) =  \ket{\psi} \otimes\ket{\psi}$. This means we cannot just send multiple copies of the quantum state through the noisy channel like the classical repetition code.

3. Destructive Measurement - In classical EC, the state can be accessed at any time, making decoding much easier. Measuring a quantum state collapses it, making the EC moot if the state is destroyed. Therefore, more clever ways to extract syndromes are required. A secondary consequence of this fact is sampling error.  Even if an algorithm could perform perfectly ensuring no sources of error, many applications require statistical sampling of the resulting state. If we sampled $\ket{\psi} = \alpha\ket{0} + \beta\ket{1}$ the frequency of 0's would be close to $\alpha^2$ but deviate based on the number of samples per the Central Limit Theorem.

4. Scalability - Though scalability is an issue for classical EC, it is far more severe for QEC. Today's noisy intermediate scale quantum devices are very difficult to control, so each additional qubit required for QEC comes at great cost.  Qubits also have short coherence times, so QEC procedures must complete within strict time constraints which gets harder at scale. Finally, the threshold theorem is in play. In classical EC, adding more bits always reduces the logical error rate. This is not true for quantum - physical qubits must have noise below a specific threshold in order for scaling the code to improve the error rates,  otherwise, the results just get worse.


## 1.5 There is still hope for QEC!

The challenges discussed above are daunting but there are many ingenious techniques developed to help circumvent them. That said, practical QEC remains difficult to realize and is an extremely active research field - viewed as one of the most important prerequisites for useful quantum computing.  This section will begin to bridge the gap between classical EC and QEC.


### Digitization of errors

Errors can perturb states incoherently from environmental effects or coherently from slight hardware imperfections.  While both types of errors can be addressed, we’ll focus on coherent errors first because they’re often easier to isolate and analyze.

For instance, a rotation gate that should be at an angle of $\frac{\pi}{16} \approx 0.196 $ ends up being more like 0.17. This may seem inconsequential, but imperfections like this accumulate and quickly ruin the outcome of a quantum algorithm.  Execute the code block below and use the slider to change the number of rotation gates executed to see how the error can become substantial.  Feel free to experiment with different values for the `angle`, `noisy_angle`, and the rotation axis in the `rotation_kernel`.




In [ ]:
# Angles of rotation of a qubit
angle = np.pi / 16
noisy_angle = 0.17

# Kernel to initialize a qubit in the zero ket state and rotate it about the x axis by given angle n times
@cudaq.kernel
def rotation_kernel(n: int, angle: float):
    qubit = cudaq.qubit()
    for _ in range(n):
        rx(angle, qubit) # CHANGE THE ROTATION AXIS

# Function to plot sample results
def plot_results(results1, results2):
    # Convert the sample results to a dictionary
    result_dictionary1 = {k: v for k, v in results1.items()}
    result_dictionary2 = {k: v for k, v in results2.items()}

    # Get all unique x-values from both dictionaries
    all_keys = set(result_dictionary1.keys()).union(set(result_dictionary2.keys()))
    all_keys = sorted(all_keys)

    # Convert the dictionary to lists for x and y values
    x1 = list(all_keys)
    y1 = list(result_dictionary1.values())
    y2 = list(result_dictionary2.values())

    # Create the combined histogram
    bar_width = 0.35
    x_indices = range(len(x1))

    plt.bar(x_indices, y1, width=bar_width, color='#76B900', label='Noise-Free Results')
    plt.bar([i + bar_width for i in x_indices], y2, width=bar_width, color='#484848', label='Noisy Results')

    # Add title and labels
    plt.title('Comparing sampling results of n applications of a noise-free gate with a noisy version')
    plt.xlabel("Basis States")
    plt.ylabel("Frequency")
    plt.xticks([i + bar_width / 2 for i in x_indices], x1)
    plt.legend()

    # Show the plot
    plt.tight_layout()
    plt.show()

# Function to update the plot based on the slider value
def update_plot(num_rotations):
    expected_result = cudaq.sample(rotation_kernel, num_rotations, angle)
    noisy_result = cudaq.sample(rotation_kernel, num_rotations, noisy_angle)
    plot_results(expected_result, noisy_result)

# Create an interactive slider
slider = widgets.IntSlider(min=1, max=20, step=1, value=1, description='n:', continuous_update=False)
interact(update_plot, num_rotations=slider)

Among the various coherent errors that can occur on a qubit storing the quantum state $\ket{\psi} = \alpha \ket{0}+\beta\ket{1}$, we will focus on three specific types:
* **Bit flip errors** swap a qubit's amplitudes, transforming $\ket{\psi}$ to $\beta\ket{0}+\alpha\ket{1}$.
* **Phase flip errors** introduce a sign change in one of the amplitudes, transforming $\ket{\psi}$ to $\alpha\ket{0}-\beta\ket{1}$.
* **Combining a bit flip with a phase flip error** swaps amplitudes and applies a sign change, transforming $\ket{\psi}$ to $\beta\ket{0}-\alpha\ket{1}$.

Run the cell below to open an interactive tool that allows you to visualize the impact of different error types on various quantum states. Observe how some error types may not alter the state. Why do you think that happens? What patterns can you identify?

In [ ]:
# Execute this cell to see the interactive widget
# Don't concern yourself with the code below this line
# Function to update and display the Bloch sphere
def update_bloch_sphere(theta, phi, error_type):
    alpha = np.cos(theta / 2)
    beta = np.sin(theta / 2) * np.exp(1j * phi)
    coefficients = [complex(alpha, 0), complex(0, beta)]
    error_types = ['Bit Flip', 'Phase Flip', 'Bit & Phase Flip']
    @cudaq.kernel
    def initial_state_kernel(coefficients: list[complex]):
        qubit = cudaq.qvector(coefficients)

    @cudaq.kernel
    def initial_state_error(coefficients: list[complex], error: int):
        qubit = cudaq.qvector(coefficients)
        if error == 0 or error == 2:
            # bit flip error
            x(qubit)
        if error == 1 or error == 2:
            # phase flip error
            z(qubit)

    state_no_error = cudaq.get_state(initial_state_kernel, coefficients)
    state_with_error = cudaq.get_state(initial_state_error, coefficients, error_type)

    blochSphereList = []
    # Define a sphere object representing the state of the single qubit
    sphere = cudaq.add_to_bloch_sphere(state_no_error)
    blochSphereList.append(sphere)
    sphere = cudaq.add_to_bloch_sphere(state_with_error)
    blochSphereList.append(sphere)

    # Create output widgets for the Bloch spheres and text
    out1 = Output()
    out2 = Output()
    text1 = Output()
    text2 = Output()

    with out1:
        cudaq.show([blochSphereList[0]], nrows=1, ncols=1)
    with out2:
        cudaq.show([blochSphereList[1]], nrows=1, ncols=1)
    with text1:
        print(f"|ψ> = cos(θ/2)|0⟩ + e^(iφ)sin(θ/2)|1⟩")
    with text2:
        print("|ψ⟩ with a ", error_types[error_type], " error")

    display(VBox([HBox([VBox([text1, out1]), VBox([text2, out2])])]))

# Create the interactive widget
theta_slider = widgets.FloatSlider(value=np.pi/2, min=0, max=2*np.pi, step=0.01, description='θ (radians):')
phi_slider = widgets.FloatSlider(value=0, min=0, max=np.pi, step=0.01, description='φ (radians):')
error_selector = widgets.Dropdown(options=[('None', -1), ('Bit Flip', 0), ('Phase Flip', 1), ('Bit & Phase Flip', 2)], value=-1, description='Error Type:')

interact(update_bloch_sphere, theta=theta_slider, phi=phi_slider, error_type=error_selector)


Once we have identified one of these errors, we can correct it. For instance, if a qubit has undergone a bit flip error, we can correct it by applying an $X$ gate. Similarly, to correct a qubit that has experienced a phase flip error, we simply apply a $Z$ gate. How would you correct a qubit that has been identified as having undergone a bit flip error followed by a phase flip error?  

We can address all coherent errors with a key insight: although the Bloch sphere suggests errors can occur through infinitely many possible rotations, all such errors can be broken down into three basic forms &mdash; bit flips, phase flips, or a combination of both bit flips and phase flips.  

If you'd like an explanation of why this decomposition works, consult the optional section below. For now, remember that by detecting and correcting these three core error types, we can effectively handle any coherent noise.  

> **Optional:** Consider a qubit in the following normalized state.
> $$ \ket{\psi}  =  \cos\frac{\theta}{2}\ket{0} + e^{i\phi}\sin\frac{\theta}{2}\ket{1} $$
>
>  Coherent errors can be represented by the application of a Unitary $U(\delta\theta,\delta\phi)$ which acts on the ideal state and perturbs it.
>$$ U(\delta\theta,\delta\phi)\ket{\psi}  =  \cos\frac{\theta +\delta\theta}{2}\ket{0} + e^{i\phi+\delta\phi}\sin\frac{\theta+\delta\theta}{2}\ket{1} $$
> Using the fact that the Pauli matrices form a basis for any 2x2 unitary matrix and taking advantage of the identity $Y=iXZ$, the operation can be rewritten as
> $$ U(\delta\theta,\delta\phi) \ket{\psi} = \alpha_II\ket{\psi} +\alpha_X X\ket{\psi}+\alpha_Z Z\ket{\psi}+\alpha_{XZ}XZ\ket{\psi}   $$
> This means that any coherent error can be **digitized** into X-type bit flip errors ($X \ket{\psi} = \alpha X\ket{0} + \beta X\ket{1} = \alpha\ket{1} + \beta\ket{0}$), Z-type phase flip errors ($Z\ket{\psi} = \alpha Z\ket{0} + \beta Z\ket{1} = \alpha\ket{0} - \beta\ket{1}$), or a combination of the two (XZ). This makes correction much more tractable, as there are only three types of errors to consider.

### Syndrome Extraction

The no cloning principle means quantum states cannot be copied for QEC. We'll need a clever way to extract syndromes from the logical state that does not rely on repetition. But, how is this done without destroying the information that is being protected?

The solution involves **stabilizers** which are specially designed operators that act on a logical state without changing it, but still enable us to learn about errors by performing projective measurement of ancilla qubits.  The next notebook in this series will introduce stabilizers with more mathematical rigor, and the example in section 1.6 of this lab will provide a more concrete example of a simple stabilizer in action.

### Better QEC codes and AI solutions

Finally, overcoming the QEC scaling challenges will require breakthroughs on many fronts. Significant research efforts are targeting discovery of more efficient QEC codes that require fewer qubits.  AI is already showing great promise as a tool to help find new QEC codes, and accelerate decoding. Later notebooks will explore AI for QEC applications.


## 1.6 The Quantum Repetition Code

A quantum state cannot be cloned, but it can be redundantly encoded across additional entangled qubits. Let's start with a generic normalized qubit state $\ket{\psi}$:

$$\ket{\psi} = \alpha\ket{0} +\beta\ket{1}.$$

The 0 and 1 states can be encoded into a logical state making use of the larger 8-dimensional Hilbert space of three qubits:  

$$\ket{\psi}_L = \alpha\ket{000} +\beta\ket{111} = \alpha\ket{0}_L +\beta\ket{1}_L.$$

Note that this is *not* equivalent to $\ket{\psi} \otimes \ket{\psi} \otimes \ket{\psi}$.

Consider now the entire Hilbert space spanned by the eight basis states.  The basis states are separated into the logical codewords that make up the codespace:   

| Codespace    | Notation for the logical codewords|
| ----------- | ----------- |
| $\ket{000}$ | $\ket{0}_L$ |
|$\ket{111}$ | $\ket{1}_L$ |

and the remaining basis states make up the error space:

| Error space |
| ----------- |
| $\ket{001}$ |
| $\ket{010}$ |
| $\ket{100}$ |
| $\ket{011}$ |
| $\ket{101}$ |
| $\ket{110}$ |

Assume that the state $\ket{111}$ is transmitted through a noisy channel and becomes $\ket{011}$?  How might it be decoded to tell which logical codeword it is closest to?  Remember, you cannot simply examine the state and perform a majority count as no information about the state is accessible without some sort of measurement that induces wavefunction collapse.  

Consider the operators $Z_1Z_2$ and $Z_2Z_3$.  (Remember that $Z_n$ returns an eigenvalue of +1 if the nth qubit in a ket is a 0 and -1 if it is a 1).  
It turns out that these operators have special properties such that the eigenvalues produced when they act on any of the states in the codespace or error space can be extracted with ancilla qubits without disturbing the state.  This means, there is a way to extract parity check information just like the classical Hamming code!

Note: Operators with these "special properties" are called stabilizers and will be rigorously introduced in the next lab.

The details of this extraction process are described shortly, but the implication is that syndromes can be produced from the parity check results and used to identify corrections to the quantum repetition code without destroying the encoded state. Considering only single bit flip errors, the table below shows the possible syndromes, the corresponding errors, and the operation that can be applied to correct the error assuming $\ket{111}$ is the transmitted message.  

| $Z_1Z_2$ Syndrome   | $Z_2Z_3$ Syndrome| Encoded State | Single Bit Flip Correction
| ----------- | ----------- | ----------- | ----------- |
| 0 | 0 | $\ket{111}$ | none |
| 1 | 0 | $\ket{011}$ | $X_1$ |
| 0 | 1 | $\ket{110}$ | $X_3$ |
| 1 | 1 | $\ket{101}$ | $X_2$ |


Like the classical repetition code, there is no way to know for certain if a 10 syndrome corresponds to a single bit flip error from the $\ket{111}$ codeword or a two bit flip error from the $\ket{000}$ codeword.  However, it is always prudent to assume that the case with fewer errors is more likely.

The entire quantum circuit for the three qubit repetition code is shown below, where the ancilla qubits are used to compute the $Z_1Z_2$ and $Z_2Z_3$ syndromes.



<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-6-images/repcircuit.png?raw=1" alt="Drawing" style="width: 800px;"/>



It is helpful to explore in greater detail how the syndromes can be extracted using the ancilla qubits without disturbing the state.
First, consider the initial state of the first ancilla after application of a Hadamard gate and the encoded state after a bit flip has occurred on the first data qubit.

$$ \frac{1}{\sqrt{2}}(\ket{0} + \ket{1})\ket{011} $$

Next, a controlled $Z_1Z_2$ operation is applied to the data qubits resulting in

$$ \frac{1}{\sqrt{2}}(\ket{0}\ket{011} + \ket{1}Z_1Z_2\ket{011}), $$

followed by an application of the second Hadamard:

$$ \frac{1}{2} ((\ket{0}+\ket{1})\ket{011} + (\ket{0} -\ket{1})Z_1Z_2\ket{011})  =  \ket{0}(\frac{1+Z_1Z_2}{2})\ket{011} + \ket{1}(\frac{1-Z_1Z_2}{2})\ket{011}  .   $$

Now, if $Z_1Z_2\ket{011}$ is evaluated, the result is $-\ket{011}$ and the entire state simplifies to $\ket{1}\ket{011}$ meaning upon measurement, the first ancilla qubit will be measured as 1 with certainty and the data qubits remain undisturbed in the $\ket{011}$ state:


$$ \ket{0}(\frac{1+Z_1Z_2}{2})\ket{011} + \ket{1}(\frac{1-Z_1Z_2}{2})\ket{011}  =   \ket{0}(\frac{1+ -1}{2})\ket{011} + \ket{1}(\frac{1--1}{2})\ket{011}  =   \ket{1}\ket{011}.  $$

A similar analysis will show that the second ancilla qubit will be measured as 0 with certainty without distubring the data qubits.  Accordoing to the syndrome table,
this will trigger an application of the $X$ gate on the first qubit to correct the error.


### Exercise 3: Coding the Quantum Repetition Code

Now that you understand the quantum repetition code, try to code it using CUDA-Q. Fill in each of the steps below marked "#TODO". CUDA-Q contains a couple of features particularly helpful for building QEC workflows. First, and already completed for you, is the definition of a custom noise model which produces custom identity operations that can randomly perform bit flips on specific qubits. Second, you can measure the ancilla qubits within the kernel and use the result to perform a correction operation. 


Try to code all the steps and then sample the kernel to determine the logical error rate.



In [ ]:
import cudaq
import numpy as np

cudaq.set_target('density-matrix-cpu')


#First, create an empty noise model
noise_model = cudaq.NoiseModel()
p = 0.1

#Build a custom gate which applies the identity operation
cudaq.register_operation("custom_i", np.array([1, 0, 0, 1]))

#Add a bitflip noise channel to the custom_i gate applied to each qubit
noise_model.add_channel("custom_i", [0], cudaq.BitFlipChannel(p))
noise_model.add_channel("custom_i", [1], cudaq.BitFlipChannel(p))
noise_model.add_channel("custom_i", [2], cudaq.BitFlipChannel(p))

@cudaq.kernel
def three_qubit_repetition_code():
    """Prepares a kernel for the 3-bit quantum repetition code

    Returns
    -------
    cudaq.kernel
        Kernel for running the 3-bit quantum repetition code

    """

    # Create register for data and ancilla qubits
    # TODO

    # Initialize the logical |1> state as |111>
    # TODO

    # Apply custom_i to induce random bitflip errors
    # TODO

    # Extract Syndromes
    # TODO

    # Correct errors based on syndromes
    # TODO

# Run the kernel and observe results
# The percent of samples that are 000 corresponds to the logical error rate
result = cudaq.sample(three_qubit_repetition_code, noise_model=noise_model)
print(result)

# Stabilizers, the Shor code, and the Steane code

This lab was motivated by content from "[Quantum Error Correction: an Introductory Guide](https://arxiv.org/abs/1907.11157)" and "[Quantum Error Correction for Dummies](https://arxiv.org/abs/2304.08678)", both excellent resources we refer readers to for additional detail.  For a more technical introduction, see chapter 10 of "[Quantum Computation and Quantum Information](https://books.google.com/books?hl=en&lr=&id=-s4DEy7o-a0C&oi=fnd&pg=PR17&dq=quantum+computation+and+quantum+information&ots=NJ4KdqnzZt&sig=uKTETo5LLjWB9F_PV_zf0Sw3bvk#v=onepage&q=quantum%20computation%20and%20quantum%20information&f=false)" or the [PhD thesis](https://arxiv.org/abs/quant-ph/9705052) where the concept of stabilizer codes was introduced.



Execute the cells below to load all the necessary packages for this lab.

In [ ]:
import cudaq
from cudaq import spin
from cudaq.qis import *
import numpy as np
import matplotlib.pyplot as plt
from typing import List

## 2.1 Stabilizers and Logical Operators


An important subclass of QEC codes, known as **stabilizer codes**, use special operations called **stabilizers** to clean up errors in encoded quantum information, and thus "stabilize" the state.


An operation $s$ acting on a state $\ket{\psi}$ is said to be a stabilizer of the state if the state is a +1 eigenstate of the operation $s \ket{\psi} = +1 \ket{\psi}$. The high-level intuiton here is that if small errors have accumulated in a logically encoded state, the action of applying this stabilizer is to project the state back to a perfectly error-free state, and we measure $+1$. Sometimes larger errors occur, and we do not measure $+1$, which informs us something has gone wrong.


In lab 1, the codespace was defined by the set of basis codewords, such as $\ket{000}$ and $\ket{111}$ for the 3-qubit quantum repetition code. In that lab the codewords were provided to you for each code, but in a stabilizer code, we can equivalently define the codespace by providing the stabilizers which stabilize each basis codeword.  In practice, this process of defining a code by the stabilizers is much more efficient and scalable as the codes grow larger.

The codespace $C$ can be defined as formed by all $\ket{\psi}$ such that $s_i\ket{\psi} = +1 \ket{\psi}$ for each $s_i\in S$, where these $s_i$ are stabilizers which form a group $S$ (note: in some texts this group $S$ is called the stabilizer, not the elements). That is, the codespace is the joint +1 eigenspace fixed by the stabilizers.  

Again in lab 1, we were given the codespace and error space for the 3-qubit quantum repetition code up front. However, let's think about working backwards from the computational basis for the 3-qubit Hilbert space. These can be sorted based on the eigenvalues returned when operated on by all elements of the stabilizer group $S = \{Z_1Z_2, Z_2Z_3\}$:

| Basis state | Eigenvalue for $Z_1Z_2$ | Eigenvalue for $Z_2Z_3$ |
| ----------- | ----------- | ---------- |
| $\ket{000}$ | 1 | 1 |
| $\ket{001}$ |  1 | -1 |
| $\ket{010}$ |  -1 | -1 |
| $\ket{100}$ |  -1 | 1 |
| $\ket{011}$ |  -1 | 1 |
| $\ket{101}$ |  -1 | -1 |
| $\ket{110}$ |  1 | -1 |
| $\ket{111}$ |  1 | 1 |




The basis states that have an eigenvalue of 1 for both $Z_1Z_2$ and $Z_2Z_3$ make up the codespace $\ket{000}$ and $\ket{111}$. There are other valid stabilizers in this code, such as $Z_1 Z_3$, but any stabilizer group can be boiled down into a minimal set which can be multiplied together to generate all of the others.

This is a really powerful approach, because it eliminates the need to derive and document the basis of the codespace in advance. Instead, one can simply define an appropriate set of stabilizers to establish the codespace.

Stabilizer codes are usually characterized as $[[n,k,d]]$ (double brackets for quantum codes) where $n$ is the number of physical qubits encoding $k$ logical qubits with distance $d$. It is always the case that these codes require $n-k$ stabilizers. The reason for this is that each stabilizer splits the original $2^n$ Hilbert space in two (the +1 and -1 eigenspace), with $2^1 = 2$ degrees of freedom remaining to define the logical qubit.

The trick then becomes finding good sets of stabilizers that correspond to QEC codes with favorable properties.

### Stabilizer Properties

Three key properties for $[[n,k,d]]$ stabilizers:

1. Here we consider only to Pauli product stabilizers, that is, $s_i$ needs to be a Pauli-group element. The n-qubit Pauli group $G_n$ is a special group constructed from the Pauli matrices:

  $$ I = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}, \quad X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}, \quad Y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}, \quad Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$$

  The group $G_n$ consists of $4*4^n$ elements and is built by forming a group that begins with all possible length $n$ Pauli words. For example $G_3$ has  terms like $IZZ$, $ZXZ$, $YYY$, etc. The group is then closed by including the possible scalar cofficients which arise from multiplying these terms $\{ 1, -1, i, -i\}$. So $G_1$ would be

  $$G_1 \equiv \{\pm I, \pm iI,\pm Z, \pm iZ,\pm X, \pm iX,\pm Y, \pm iY\}$$


2. Each $s_i$ must be able to operate on every logical state $\ket{\psi_L}$. Furthermore, this action should leave each $\ket{\psi_L}$ fixed (i.e., the eigenvalue of $\ket{\psi_L}$ should be +1 for each logical state).

3. Stabilizers need to be measurable in any order. This means each stabilizer needs to **commute** with every other stabilizer (that is, $s_is_j = s_js_i$, or equivalently $[s_i, s_j] \equiv s_is_j - s_js_i=0$).


### Logical Operators

In addition to stabilizers, each $[[n,k,d]]$ code has $2k$ **logical operators** ($\bar{X}_i$ and $\bar{Z}_i$) which perform $X$ and $Z$ operations on the logical states. For example:

$$ \bar{Z}_2\bar{X}_1\ket{11}_L = \bar{Z}_2\ket{01}_L = -\ket{01}_L $$

These logical operators must satisfy two properties.

1. $\bar{X}_i$ and $\bar{Z}_i$ commute with all stabilizers.
2. $\bar{X}_i$ and $\bar{Z}_i$ must **anticommute** with one another if acting on the same logical qubit (i.e.,  $\{\bar{X}_i,\bar{Z}_j\}\equiv \bar{X}_i\bar{Z}_j + \bar{Z}_j\bar{X}_i= 2\delta_{ij}I$ for all $i,j$).


The next few sections will make use of stabilizers to solidify these concepts and enable coding of QEC codes far more interesting than the quantum repetition code.

## 2.2 The Steane Code

The Steane code is a famous QEC code that is the quantum version of the [7,4,3] Hamming code introduced in the first part.  One immediate difference is that the Steane code encodes a single logical qubit making it a [[7,1,3]] code.

Remember, that the Hamming code adds additional parity bits that help "triangulate" where an error occurred. In the previous exercises you constructed the generator matrix $G$ and used it to produce the logical codewords in the classical Hamming code. For example, $b=0110$ was encoded as


$$
c = bG=
 \begin{bmatrix} 0 & 1 & 1 & 0 \end{bmatrix}
\cdot
\begin{bmatrix}
1 & 0 & 0 & 0 & 1 & 1 & 0 \\
0 & 1 & 0 & 0 & 1 & 0 & 1 \\
0 & 0 & 1 & 0 & 0 & 1 & 1 \\
0 & 0 & 0 & 1 & 1 & 1 & 1
\end{bmatrix}
=
\begin{bmatrix}
  0 & 1& 1 & 0 & 1& 1& 0
\end{bmatrix}
$$

Any logically encoded state, $c$, could then be multiplied by the parity check matrix ($H$) to determine if any syndromes were triggered or not.


$$
Hc^T
=
\begin{bmatrix}
1 & 1 & 0 & 1 & 1 & 0 & 0 \\
1 & 0 & 1 & 1 & 0 & 1 & 0 \\
0 & 1 & 1 & 1 & 0 & 0 & 1
\end{bmatrix}
\cdot
\begin{bmatrix}
0 \\
1 \\
1 \\
0 \\
1 \\
1 \\
0
\end{bmatrix}
=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}.
$$

What was not discussed in the Hamming code section was the fact that the parity check matrix ($H$) can be used to define the codespace. A valid logical codeword is any $c$ that satisfies the relationship $Hc^T=\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}$.  As there are 7 data bits, that means there are $2^7=128$ possible encoded states between the codespace and error space.  It turns out, 16 of these fall within the codespace and 112 are in the error space. Of the 16 in the codespace, 8 have even parity (even number of 1's) while the other half has odd parity.

| Even Bitstrings in Codespace | Odd Bitstrings in Codespace |
| ----------- | ----------- |
| 0000000 | 1111111 |
| 0001111 | 1110000 |
| 0110110 | 1001001 |
| 0111001 | 1000110 |
| 1010101 | 0101010 |
| 1011010 | 0100101 |
| 1100011 | 0011100 |
| 1101100 | 0010011 |


This provides us a way to define the logical code words: $\ket{0}_L$ and $\ket{1}_L$.
The logical codewords, $\ket{0}_L$ and $\ket{1}_L$, for the Steane code are superpositions over the states corresponding to the classic even and odd codewords, respectively.



$$ \ket{0}_L = \frac{1}{\sqrt{8}}(\ket{0000000} +\ket{0001111} +\ket{0110110} + \ket{ 0111001} + \ket{1010101} + \ket{1011010}+ \ket{1100011} + \ket{1101100})  $$

$$ \ket{1}_L = \frac{1}{\sqrt{8}}(\ket{1111111} +\ket{1110000} +\ket{1001001} + \ket{1000110} + \ket{0101010} + \ket{0100101}+ \ket{0011100} + \ket{0010011})  $$


You might notice by inspection, that $\ket{0}_L = \bar{X}\ket{1}_L = X_1X_2X_3X_4X_5X_6X_7\ket{1}_L$.  That is to say, flipping all the bits swaps logical states.   Similarly, $\bar{Z} =  Z_1Z_2Z_3Z_4Z_5Z_6Z_7$ will flip the phase, transforming $\frac{1}{\sqrt{2}}(\ket{0}_L+\ket{1}_L)$ to $\frac{1}{\sqrt{2}}(\ket{0}_L-\ket{1}_L)$

The encoding circuit to produce the logical codewords is shown below, and is based off the constraints imposed by the parity check matrix.

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-6-images/steaneencoding.png?raw=1" alt="Drawing" style="width: 500px;"/>







In [ ]:
### Exercise  4 - The Steane Code:

In the cell below, build a CUDA-Q kernel to encode the logical 0 state using the Steane code.  Sample the circuit to prove that you indeed created the appropriate superposition.  In the cells following, complete the entire Steane code by adding stabilizer checks and code to measure the logical state. Complete the numbered tasks as well to confirm your code works as expected.


In [ ]:
@cudaq.kernel
def steane_code():
    """Prepares a kernel for the Steane Code
    Returns
    -------
    cudaq.kernel
        Kernel for running the Steane code
    """

    #Initialize Registers
    #TODO

    # Create a superposition over all possible combinations of parity check bits
    #TODO

    #Entangle states to enforce constraints of parity check matrix (circuit above)
    #TODO



results = cudaq.sample(steane_code, shots_count=10000)
print(results)

The Steane code is a member of an important family of stabilizer codes known as **Calderbank-Shor-Steane (CSS)** codes. A CSS code is characterized by the property that Z and X errors can be detected and corrected independently.  A benefit of this, is that fewer ancilla qubits are required to produce the syndromes, and they can be reset and reused for each error. However, this procedure is also slower.

The stabilizers for $Z$-type errors are $S_Z = \{X_1X_2X_5X_4, X_1X_3X_4X_6, X_2X_3X_4X_7\},$ while the stabilizers for $X$-type errors are of similar form: $S_X = \{Z_1Z_2Z_5Z_4, Z_1Z_3Z_4Z_6, Z_2Z_3Z_4Z_7\}$.

Just as with the classical Hamming code, there is a nice way to visualize the syndrome results. The diagram below places each data qubit on each vertex. They are arranged such that each of the three sections or **plaquettes** corresponds to one of the stabilizers.  

The syndromes can be visually interpreted by putting a colored X on the syndromes that are flagged. Each coloring of this graph uniquely corresponds to an error on a specific qubit which is why the Steane code is often referred to as a **color code**.


<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-6-images/plaqettes.png?raw=1" alt="Drawing" style="width: 700px;"/>

You are now ready to code the rest of the Steane code.  After encoding, introduce an $X$ error and $Z$ error on the qubits of your choice.  Try performing the $X$ and $Z$ syndrome measurements using the same three ancilla qubits and resetting them in between. Make your code such that you can measure the data qubits and confirm the state of the logical qubit.  

In [ ]:
import cudaq
@cudaq.kernel
def steane_code():
    """Prepares a kernel for the Steane Code
    Returns
    -------
    cudaq.kernel
        Kernel for running the Steane code
    """

    #Initialize Registers
    #TODO

    # Create a superposition over all possible combinations of parity check bits
    #TODO

    #Entangle states to enforce constraints of parity check matrix (circuit above)
    #TODO

    #Add Errors (Optional)
    #TODO



    # Perform Stabilizer checks for Z errors
    #TODO


    # Perform Stabilizer checks for X errors
    #TODO


    # Correct X errors
    #TODO

    # Correct Z errors
    #TODO


results = cudaq.sample(steane_code, shots_count=1000)
print(results)

#Post-process Results
#TODO

Now, test your code! Just measure in the $Z$ basis as the same procedure could be performed with the $X$ basis.

1. Try adding single $X$ errors, guess which stabilizers should flag and confrm they do.
2. Add two errors. Confirm the code cannot correct the errors and a logical bitflip occurs.
3. It turns out that like the Shor code, there are alternate choices for $\bar{X}$.  Modify your counting code above and test if $X_0X_1X_4$ or $X_0X_4X_5$ are valid choices for $\bar{X}$.  

## 2.3 Steane Code Capacity Analysis with CUDA-Q QEC


[CUDA-QX](https://developer.nvidia.com/cuda-qx) is set of libraries that enable easy acceleration of quantum application development.  One of the libraries, [CUDA-Q QEC](https://nvidia.github.io/cudaqx/components/qec/introduction.html), is focused on error correction and can help expedite much of the work done above.  This final section will demonstrate how to run a code capacity memory experiment with the Steane code.

A memory experiment is a procedure to test how well a protocol can preserve quantum information. Such an experiment can help assess the quality of a QEC code but is often limited by assumptions that deviate from a realistic noise model. One such example is a code capacity experiment. A code capacity procedure determines the logical error rate of a QEC code under strict assumptions such as perfect gates or measurement.  Code capacity experiments can help put an upper bound on a procedure's threshold and is therefore a good starting place to compare new codes.

The process is outlined in the diagram below.  Assume the 0000000 bitstring is the baseline (no error).  Bitflips are then randomly introduced and produce errors in the data vector to produce results like 0100010.  If this were a real test on a physical quantum device, the data vector would not be known and a user could only proceed through the bottom path in the figure - performing syndrome extraction and then decoding the result to see if a logical flip occurred. In a code capacity experiment, the data vector with errors is known, so it can be used to directly compute if a logical state flip occurred or not.  Dividing the number of times the actual (top path) and predicted (bottom path) results agree by the total number of rounds provides an estimate of the logical error rate for the code being tested.


<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-6-images/steanecodecapacity.png?raw=1" alt="Drawing" style="width: 700px;"/>






In [ ]:
### Exercise  5 - CUDA-Q QEC Code Capacity Experiment:

CUDA-Q QEC allows researchers to streamline experiments like this with just a few lines of code.  Try running the cells below to compute the logical error rate of the Steane code under code capacity assumptions given probability of error $p$.


In [ ]:
import numpy as np
import cudaq_qec as qec

Next, load the Steane code, which is already implemented in CUDA-Q QEC.

In [ ]:
steane = qec.get_code("steane")

The parity check matrices and observables can also be extracted from the Steane code.

In [ ]:
Hz = steane.get_parity_z()
Hx = steane.get_parity_x()
H = steane.get_parity()
observable  = steane.get_observables_z()

A decoder can then be specified which takes the parity check matrix as an input.

In [ ]:
decoder = qec.get_decoder("single_error_lut", Hz)

Then, `sample_code_capacity` can be called and provided with `p`, the probability of any bit flipping, and the number of shots for the analysis.

In [ ]:
p = 0.1 # set a probability of a bit flip error occuring
nShots = 100 # specify the number of shots
syndromes, data = qec.sample_code_capacity(Hz, nShots, p)

for x in range(nShots):
    print("Data Qubits", data[x], "Syndromes", syndromes[x])

Notice how the Steane code is already defined within CUDA-Q QEC, along with a selection of decoders, and the `sample_code_capacity` API to automatically run the procedure.  Otherwise, you would need to code the entire process from scratch like you did in section 2.2 for each QEC code you want to test!

If the experiment is repeated many times with different $p$ values, a plot can be generated like the one shown below. The purple line is the $y=x$ and corresponds to the case that the logical error rate is identical to the physical error rate.  Anywhere the green line is below the purple line indicates that the Steane code was able to produce a logical error rate that is less than the physical error rate of the data qubits. When the green line is above the purple, the Steane code produced a worse logical error rate indicating that it would have been better to just use the data qubits and avoid the QEC procedure. The crossover point is an estimate for the code's threshold. Refining this estimate would require more sophisticated circuit level noise models that more accurately represent the performance of the Steane code under realistic conditions.

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-6-images/codecapacityplot.png?raw=1" alt="Drawing" style="width: 700px;"/>

Though code capacity has much room to improve, it is a great example of the utility of CUDA-Q QEC and how simple procedures can be streamlined so users can focus on testing codes rather than coding up the details of each test.

## 2.4 The Shor Code

The first QEC code was proposed by Peter Shor in 1995, known as the [Shor code]((https://journals.aps.org/pra/abstract/10.1103/PhysRevA.52.R2493)).  The Shor code is a [[9,1,3]] code which uses 9 qubits to encode a single qubit, but can correct single $X$ or $Z$-type errors.


The motivation for the code, is that the 3-qubit repetition code can correct bit flip errors but not phase flip errors.  We can consider why this is by examining the encoded $\ket{+}_L$ state, which looks like the following:

$$ \ket{+}_L = \frac{1}{\sqrt{2}}(\ket{0}_{L~3\mathrm{bit}}+\ket{1}_{L~\mathrm{3bit}}) =  \frac{1}{\sqrt{2}}(\ket{000}+\ket{111}) $$

If a $Z_1$, $Z_2$, or a $Z_3$ error occurs, the $ \ket{+}_L$ state is transformed to $ \ket{-}_L$, another valid codeword.  This means there is no way to tell if a phase flip error occurred or not. One could produce the repetition code in the  $ \ket{+}_L$ and $ \ket{-}_L$ basis to correct $Z$ errors, but then the same problem would persist for $X$ errors.

The ingenuity behind the Shor code is to concatenate two 3-bit repetition codes into a 9-qubit code that can detect both types of errors. The encoding process begins with the 3-bit encoding of the $\ket{+}$ state.

$$ \ket{+}_{\mathrm{3 bit}} = \frac{1}{\sqrt{2}}(\ket{000}+\ket{111})$$

Then, $\ket{0}_L$ is encoded by taking a tensor product of three $\ket{+}_{\mathrm{3 bit}}$ states.


$$ \ket{0}_L = \ket{+}_{\mathrm{3 bit}} \otimes \ket{+}_{\mathrm{3 bit}} \otimes \ket{+}_{\mathrm{3 bit}} $$
$$ \ket{0}_L = \frac{1}{\sqrt{8}}(\ket{000} + \ket{111})(\ket{000} + \ket{111})(\ket{000} + \ket{111})$$

The same process is completed for the $\ket{1}_L$ state, this time using $\ket{-}_{\mathrm{3 bit}}$ as the starting point.
$$ \ket{1}_L = \frac{1}{\sqrt{8}}(\ket{000} - \ket{111})(\ket{000} - \ket{111})(\ket{000} - \ket{111})$$

This encoding of $\psi = \alpha \ket{0} + \beta \ket{1}$ can be implemented with the following quantum circuit:

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-6-images/shorencode.png?raw=1" alt="Drawing" style="width: 300px;"/>


The next consideration is to define the logical operators so that they behave as we expect, namely: $$\bar{X}\ket{0}_L = \ket{1}_L $$
$$\bar{X}\ket{1}_L = \ket{0}_L $$
and
$$\bar{Z}\ket{0}_L = \ket{0}_L$$
$$\bar{Z}\ket{1}_L = -\ket{1}_L.$$
In other words, we need the following equations to hold:

$$\bar{X}\ket{0}_L = \bar{X}\frac{1}{\sqrt{8}}(\ket{000} + \ket{111})(\ket{000} + \ket{111})(\ket{000} + \ket{111}) = \frac{1}{\sqrt{8}}(\ket{000} - \ket{111})(\ket{000} - \ket{111})(\ket{000} - \ket{111}) = \ket{1}_L  $$


$$\bar{Z}\ket{1}_L = \bar{Z}\frac{1}{\sqrt{8}}(\ket{000} - \ket{111})(\ket{000} - \ket{111})(\ket{000} - \ket{111}) = \frac{1}{\sqrt{8}}(\ket{111} - \ket{000})(\ket{111} - \ket{000})(\ket{111} - \ket{000}) = -\ket{1}_L$$


Can you see what the logical operators need to be?  


For a logical bit flip to occur ($\bar{X}$) the phase of each block needs to change.  This is accomplished by performing a $Z $ operation on one of the qubits in each block, thus $\bar{X} = Z_1Z_4Z_7$ is a valid choice, though not the only choice as others like $\bar{X} = Z_2Z_5Z_8$ or even $\bar{X} = Z_1Z_2Z_3Z_4Z_5Z_6Z_7Z_8Z_9$ also work. Similarly, for $\bar{Z}$ to take $\ket{1}_L$ to $-\ket{1}_L$ (and $\ket{0}_L$ to itself) all of the bits need to flip, thus  $\bar{Z} = X_1X_2X_3X_4X_5X_6X_7X_8X_9$. The curious reader can confirm that the anticommutativity holds between these logical operators and that they commute with each stabilizer discussed below.


Now, consider what happens when $X$ and $Z$-type errors corrupt a state encoded with the Shor code. If a bitflip error occurs on qubit 8 of the $\ket{0}_L$ state.

$$ X_8\ket{0}_L = \frac{1}{\sqrt{8}}(\ket{000} + \ket{111})(\ket{000} + \ket{111})(\ket{010} + \ket{101})$$

The error only corrupts the third block of the code which houses the 8th qubit. So, this means stabilizers that perform parity checks on that block only  ($Z_7Z_8$ and $Z_8Z_9$) are sufficient to determine which position experienced an error. Extending this, all bit flip errors can be corrected with the following six stabilizers, two for each block.  Because each block is an independent repetition code, the Shor code can handle three bit flip errors,  as long as they occur in distinct blocks.

$$ S_{\mathrm{bit flips}} = \{Z_1Z_2, Z_2Z_3, Z_4Z_5, Z_5Z_6, Z_7Z_8, Z_8Z_9\} $$

Now consider the impact of a a phase flip error that acts on the 6th qubit, for example.

$$ Z_6\ket{0}_L = \frac{1}{\sqrt{8}}(\ket{000} + \ket{111})(\ket{000} - \ket{111})(\ket{000} + \ket{111})$$

The phase of the second block is changed which means the entire state can be rewritten as $\ket{+}_{\mathrm{3 bit}} \otimes \ket{-}_{\mathrm{3 bit}} \otimes \ket{+}_{\mathrm{3 bit}}$.  This "zoomed out" view makes it clear how the repetition code is leveraged again. This time a stabilizer is needed which can test the parity of block 1 with block 2 and block 2 with block 3.

The stabilizer $X_1X_2X_3X_4X_5X_6$ can be used for this which will return 1 if the first two blocks have the same phase and -1 if they differ. Work this out by hand to convince yourself this works if it is not obvious why this is the case. Similarly, $X_4X_5X_6X_7X_8X_9$ can test the parity of the second two blocks completing the stabilizers necessary to detect phase flip errors.

$$ S_{\mathrm{phase flips}} = \{X_1X_2X_3X_4X_5X_6,X_4X_5X_6X_7X_8X_9   \} $$

All 8 stabilizers can correct any single-qubit $Z$ or $X$ error as summarized in the table below. Note that the Shor code is a redundant code, meaning that certain syndromes correspond to multiple errors.  At first this may seem problematic, but each error is fixed by the same correction, so knowing the specific source of the error is not always necessary.

| Error Type | Syndrome (Stabilizer Measurements) |
| ----------- | ----------- |
| No Error | 0 0 0 0 0 0 0 0 |
| $X_1$ | 1 0 0 0 0 0 0 0 |
| $X_2$ | 1 1 0 0 0 0 0 0 |
| $X_3$ | 0 1 0 0 0 0 0 0 |
| $X_4$ | 0 0 1 0 0 0 0 0 |
| $X_5$ | 0 0 1 1 0 0 0 0 |
| $X_6$ | 0 0 0 1 0 0 0 0 |
| $X_7$ | 0 0 0 0 1 0 0 0 |
| $X_8$ | 0 0 0 0 1 1 0 0 |
| $X_9$ | 0 0 0 0 0 1 0 0 |
| $Z_1$ | 0 0 0 0 0 0 1 0 |
| $Z_2$ | 0 0 0 0 0 0 1 0 |
| $Z_3$ | 0 0 0 0 0 0 1 0 |
| $Z_4$ | 0 0 0 0 0 0 1 1 |
| $Z_5$ | 0 0 0 0 0 0 1 1 |
| $Z_6$ | 0 0 0 0 0 0 1 1 |
| $Z_7$ | 0 0 0 0 0 0 0 1 |
| $Z_8$ | 0 0 0 0 0 0 0 1 |
| $Z_9$ | 0 0 0 0 0 0 0 1 |





<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px; border-radius: 4px;">
    <h3 style="color: #76b900; margin-top: 0;"> Exercise  3 - The Shor Code:</h3>
    <p style="font-size: 16px; color: #333;">
Now you have all of the backgound necessary to code the Shor code in CUDA-Q.  Fill in the sections below to build up a kernel that performs Shor code encoding and syndrome checks. The kernel should be constructed such that you can apply errors and select mesurement in the $Z$ or $X$ basis. Complete the tasks listed below to ensure your code works.
    </p>
</div>


In [ ]:
import cudaq

@cudaq.kernel
def shor_code(error_qubit: list[int], error_location: list[int], measure: int):
    """Prepares a kernel for the Shor Code

    Parameters
    -----------
    error_qubit: list[int]
        a list where each element is an applied error designated as 1 =x or 2 =z
    error_location: list[int]
        each element corresponds to the index of the qubit which the error occurs on
    measure: int
        Option to measure in the z basis (0) or the x basis (1)

    Returns
    -------
    cudaq.kernel
        Kernel for running the Shor code
    """

    #Encode the data qubits with Shor encoding circuit.  Hint: It might be helpful to create separate registers for the data and ancilla qubits
    #TODO

    # Initial Psi (25/75) distribution in Z and X basis
    ry(1.04772,data_qubits[0])
    rz(1.521, data_qubits[0])

    # Apply optional single qubit errors
    #TODO

    # Apply Hadamard gate to ancilla qubits
    #TODO

    # Apply the Bit Flip syndromes
    #TODO

    # Apply the phase flip syndromes
    #TODO

    # Apply Hadamard gate to ancilla qubits
    #TODO

    # Perform mid-circuit measurements to determine syndromes
    #TODO


    # Apply the appropriate corrections based on the results from the syndrome measurements
    #TODO


    #Perform Hadamard on data qubits to rotate out of X basis (because of concatonated code)
    #TODO

    #Measure in X or Z basis depending on kernel input
    h(data_qubits) # put a Hadamard before the measurement to transform back into the Z basis
    # An X basis measurement can be obtained by applying a second Hadamard before a Z basis measurement
    #TODO

The final Hadamard is necessary because the code is concatenated and the second layer is in the $X$ basis.  Specifying the measurement basis allows you to confirm that the errors were or were not fixed.

You will also need to post process the results.  In the case of $Z$ basis measurement (where you see the impact of logical $X$ errors), you need to compute the parity of the logical $X$ operator $Z_1Z_2Z_3Z_4Z_5Z_6Z_7Z_8Z_9$, by measuring all the qubits in the $Z$ basis and computing the parity (Sum them and then mod 2) of the results.  

The same can be done for an $X$ basis measurement (where you see the impact of logical $Z$ errors). In this case you need to compute the parity of the logical $Z$ operator $X_1X_2X_3X_4X_5X_6X_7X_8X_9$ by measuring all of the qubits in the $X$ basis and computing thier parity.

Write a postprocessing function below which takes results, computes the parity of each measurment, prints the number of 1's and 0's, and prints the results.

In [ ]:
def post_process(results):
    """takes results from a CUDA-Q sample and prints the results and the number of 0's and 1's by computing the parity of the bitstrings.

    Parameters
    -----------
    results: cudaq.SampleResult
                A dictionary of the results from sampling the quantum state
    """


Now, run your code through the following tests and confirm it is working well.

1. Prepare $\ket{\psi}$ in the $\ket{0}$ state.  Sample your kernel with no errors in the $Z$ and $X$ basis.  Do you see a 100/0 and 50/50 distribution for each respectively? What do you notice about the bitstrings when you measure in each basis?

2. Now, comment out the part of your code that fixes errors. Add a single $Z$ error and measure in the $Z$ and $X$ basis. Do you observe a bitflip in the $Z$ basis results?  Note, because the Shor code has an extra layer of Hadamard gates, a $Z$ error impacts the $Z$ observable which is not the usual case.

3. Now prepare $\ket{\psi}$ in the $\ket{+}$ state. Comment out the part of your code that fixes errors, add a single $X$ error, and measure in the $Z$ and $X$ basis. Do you observe a bitflip in the $X$ basis results now?

4. Uncomment the part of your code that fixes the errors and run the same samples in 2 and 3. Did the correct syndrome flag and was the impact of the error ameliorated?

5. Prepare $\ket{\psi}$ in the $\ket{0}$ state again. With your full code, add multiple $Z$ errors and note that the stabilizer checks cannot properly fix them as the code is only distance 3.

In [ ]:
# TODO
# Run tests on your Shor Code

#  Simulating Quantum Noise 

In [ ]:
import cudaq
import numpy as np
import cudaq_solvers as solvers
from typing import List, Optional
import matplotlib.pyplot as plt
from cudaq import spin, operators, ScalarOperator, Schedule, ScipyZvodeIntegrator
import cupy as cp
import os

## 3.1 Quantum Noise Channels ##

In the first lab of this series, the concept of a **noise channel** was introduced. A noise channel is a mathematical model used to describe how a quantum state is impacted by the presence of noise.

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-6-images/nc.png?raw=1" alt="Drawing" style="width: 800px;"/>

A noise channel can correspond to application of a gate to physical qubits, a qubit's interaction with another nearby qubit, or simply the passage of time and the resulting decay of the quantum state as it interacts with anything else from the environment. QEC is a promising solution to this problem as a logically encoded quantum state can go through the noise channel, impacting each data qubit, while providing a means for the original state to be restored.

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-6-images/nc_qec.png?raw=1" alt="Drawing" style="width: 1100px;"/>

However, as previous labs have emphasized, QEC is hard to implement, and the development of new QEC protocols is still an active research field.  In practice, experimental data obtained from the QPU can help measure quantities like gate fidelity and inform a **noise model** which captures all of the noise channels present in the device. This noise model can then be used to simulate data that emulates the performance of the QPU.

There are many practical benefits to this that will be explored in this lab. A recent example of this is [NVIDIA's work with QuEra](https://developer.nvidia.com/blog/nvidia-and-quera-decode-quantum-errors-with-ai/) to build an AI decoder.  Training this model required a massive amount of data which could be obtained efficiently via simulation.  Noisy circuit simulation allowed for millions of syndromes to be obtained with their associated errors, something not possible to do with experimental data.


### The Density Matrix ###

Before discussing some of the ways to simulate noise, it is necessary to take a step back and consider representation of a quantum state using the **density matrix**.  The density matrix ($\rho$) is a mathematical object that completely describes a quantum state and has the following properties.

1. Its trace (sum of the diagonal elements) is equal to 1.
2. It is Hermitian: $\rho = \rho ^{\dagger}$
3. It is positive semi-definite. (All eigenvalues are positive.)

If a quantum system is in one of a any quantum states $\ket{\psi_i}$ with probability $p_i$, then the density matrix is defined as a linear combination of outer products of those states with probability coefficients:

$$\rho = \sum_i p_i \ket{\psi_i}\bra{\psi_i} $$








In [ ]:
### Exercise  6:
use CUDA-Q's $\texttt{get\_state}$ function and the density matrix simulator (more on that later) to produce any three qubit density matrix.  Write code to check that the three properties listed above are met. Make sure to set tolerances on these checks so that, for example, an eigenvalue of zero is not wrongfully flagged as `-1.2e-20`.


In [ ]:
import cudaq
import numpy as np

cudaq.set_target("density-matrix-cpu")

#Build Kernel and get state
#TODO

#get density matrix
#TODO

# Compute Trace
#TODO

# Check if Hermitian
#TODO

# Check if positive semi-definite
#TODO



<br>
<br>

Statevectors correspond to **pure states**, while the density matrix can describe **mixed states**, that is an overall state composed of a combination of pure states.

A state is considered pure if the trace of $\rho^2$ is equal to 1.

This can be a bit confusing because a pure state can actually be a superposition state and a mixed state can be a combination of two states that do not describe superpositions.  The following exercise will make this more clear.






In [ ]:
### Exercise  7 :
Consider the density matrix $\rho = \frac{1}{2}\ket{00}\bra{00} + \frac{1}{2}\ket{11}\bra{11}$.  

Using CUDA-Q build kernels for the $\ket{00}$ state and the $\ket{11}$ state, using these kernels and the $\texttt{get\_state}$ command define the density matrix $\rho$, and compute trace($\rho^2$).  Is the state pure?
   

In [ ]:
#TODO


Now, code a Bell state and do the same thing with its density matrix.  Is it a pure state?

In [ ]:
#TODO


<br>
<br>

A mixed state means that there is classical uncertainly about which quantum state defines the system, even if both quantum states are deterministic like $\ket{00}$ and $\ket{11}$.   However, a bell state is pure, meaning that the overall quantum state is known with certainly, even if the state describes a superposition with inherent uncertainty.  Another key term is **completely mixed state**, which refers to a density matrix where all of the eigenvalues are the same, meaning the density matrix describes the state with the theoretical maximum of uncertainty.

### Kraus Operators ###

Now, why the business about density matrices?  The answer is that a noise channel needs to  be an effective model that can generalize to impact mixed states. In fact, many noise channels will produce a mixed state from a pure state.

Mathematically this is done with **Kraus operators** ($K_i$) that evolve the density matrix as the state proceeds through a noisy channel $\epsilon$.

$$ \epsilon(\rho) = \sum_i K_i\rho K_i^{\dagger} $$

Kraus operators have the condition that $ \sum_i K_i K_i^{\dagger} =1 $ so the trace of the density matrix is preserved.

For example, a valid set of operators is $K_0 = \sqrt{1-p} I $ and $K_1 = \sqrt{p}X$ which will perform a bitflip error with probability $p$ and apply the identity (no change) with probability $1-p$. Let's apply this to the density matrix, $\rho_0$, for the $\ket{0}$ state. The result becomes $ \epsilon(\rho_0) = (1-p)I\rho_0 I + pX\rho_0 X $. Notice the result is now mixed state.

The table below summarizes some of the channels included in CUDA-Q which you will use in later exercises.  Notice too, that each noise channel can be geometrically represented as a deformation of the Bloch sphere.


<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-6-images/channels.png?raw=1" alt="Drawing" style="width: 800px;"/>



By applying any number of Kraus operators to the density matrix, it is possible to evolve it and sample the resulting state to determine how noise impacts the output. This is easily accomplished in CUDA-Q with the `density-matrix-cpu` backend.  You can then build a noise model consisting of noisy channels applied to specific gate operations with select probabilities. The exercise below will get you started with the syntax.






In [ ]:
### Exercise  8 :
    
You will be using CUDA-Q's built in noise channel tools throughout this lab.  Get a sense for how it works by building a two qubit kernel and perform an $X$ operation on each qubit.  Edit the code block below to build a noise model consisting of two bitflip channels with probabilities of .10 and .25 on the $X$ gate for qubit 0 and 1, respectively.  Does the sample distribution agree with what you would expect?
    

In [ ]:
noise = cudaq.NoiseModel() # #Defines an empty noise model

noise.add_channel('x', [0], cudaq.BitFlipChannel(.1)) # Adds a bitflip error channel (p=0.1) to X gates on qubit 0.
#TODO Add noise channel for second qubit

@cudaq.kernel
def test():
    reg = cudaq.qvector(2)
    x(reg)


print(cudaq.sample(test, noise_model=noise))

## 3.2 Two Ways to Simulate Noise ##

Density matrix simulation can produce exact results with the quality of simulation limited only by the accuracy of the underlying noise model.  Unfortunately, density matrix simulation is expensive and requires storage of the entire $2^N \times 2^N $ matrix, limiting it to a smaller number of qubits.

This scalability problem can be circumvented with a method called trajectory based simulation which allows for approximate noise simulation at much larger scales.  Unlike density matrix simulation that applies Kraus operators to every state, trajectory based simulation assumes the Kraus operators occur as a Markov process.

The assumption of a Markov process is that the application of each Kraus operator is independent from the others. This is usually a reasonable assumption as a physical QPU might, for example, only apply gates in an isolated gate zone.

The code blocks below will make it clear how the two approaches differ. Consider a very basic circuit that prepares the $\ket{111}$ state with bitflip errors on each qubit corrupting the result. First, run the cell below.  Notice that `get_state` returns the same density matrix each time you run the code.  This density matrix describes the mixture of all possible pure states and returns the sample distribution below.

In [ ]:
import cudaq

cudaq.set_target("density-matrix-cpu")

noise = cudaq.NoiseModel()
for q in range(3):
    noise.add_channel('x', [q], cudaq.BitFlipChannel(.2))

@cudaq.kernel
def test():
    reg = cudaq.qvector(3)
    x(reg)


cudaq.set_noise(noise)
for x in range(2):
    print(cudaq.get_state(test))


print(cudaq.sample(test, noise_model = noise))

Trajectory based simulation can run in CUDA-Q by simply changing the target to `nvidia`.  If the kernel below had no noise, the statevector (output from `get_state`) should be [0,0,0,0,0,0,0,1] corresponding to the $\ket{111}$ state. When sampling is performed with the trajectory based simulator, the Kraus operators are applied based on their probabilities to produce a new state vector for each shot. The widget below allows you to explore the possible outcomes and their associated probabilities.   

Try running the CUDA-Q simulation above with two or three different bitflip error probabilities and set the slider below to match. Confirm that the results from the density matrix simulations above match the expected distribution from the trajectory-based approach. You will need to move the `hands-on-6-res > trajectory_widget.py` file from the github repo into your working directory to execute this optional cell.

In [ ]:
from trajectory_widget import show_error_tree_widget

# this will render the entire widget
show_error_tree_widget()

Running the code below, notice `get_state` produces a different state vector each time. Because the number of possible trajectories is small, trajectory based sampling can reproduce the same distribution that would be obtained from density matrix simulation.

⚠️

Just a heads-up: The rest of this notebook is designed to be run on an environment with a GPU. If you don't have access to a GPU, feel free to read through the cells and explore the content without executing them. Enjoy learning!

⚠️

In [ ]:
import cudaq

cudaq.set_target("nvidia")

noise = cudaq.NoiseModel()
for q in range(3):
    noise.add_channel('x', [q], cudaq.BitFlipChannel(.2))

@cudaq.kernel
def test():
    reg = cudaq.qvector(3)
    x(reg)

cudaq.set_noise(noise)
for x in range(20):
    print(cudaq.get_state(test))

print(cudaq.sample(test))

Another benefit of trajectory based simulation is that it can be used with tensor network based simulators to simulate circuits that would be far too large for density matrix or statevector simulation. CUDA-Q can run exact tensor network or approximate Matrix Product State (MPS) simulations with trajectory based simulation to simulate systems of hundreds to thousands of qubits.

Clever sampling algorithms can also be used to filter trajectories and exclude certain types of errors or focus on sampling only a subset of the most likely errors.  A [recent paper published by NVIDIA research]()https://arxiv.org/pdf/2504.16297 explains this in greater detail and described how methods like this can sample trillions of noisy samples in just a few hours using an AI supercomputer. This is extremely helpful for training AI QEC decoders where experimental data cannot be obtained in sufficient volume.

 ## 3.3 Use cases for Noisy Simulations ##

This section will explore three use cases of noisy simulation used to model the impact of noise patterns on algorithms, perform quantum error mitigation, and run QEC experiments.

### 3.3a: Understanding How Noise Impacts Algorithm Results ##3

A natural application of noisy simulation is to explore how different noise patterns might impact the results of an algorithm.  Such simulations can be beneficial for a number of reasons. This section along with the following two will explore three use cases for noisy circuit simulation.

The first use case is to better understand how device noise impacts the outcome of an algorithm. Researchers can use these sorts of results to produce insights that guide compilation methods by identifying how particular algorithms might be more or less sensitive to particular noise channels. Such an approach is also useful to develop noise models by tuning them to agree with the results obtained running the same application experimentally.

This section will walk you through an exercise to explore the impact of noise on a standard chemistry experiment.  The CUDA-Q Solvers library makes it easy to prepare a quantum circuit to compute the ground state energy of a molecule. The code section below prepares a circuit with the standard UCCSD ansatz as well as the Hamiltonian of the hydrogen molecule. Run the cell below to get the noiseless energy and see a print out of the circuit.  Note: the circuit parameters are not optimized, but this does not matter as it is just a reference point to study the impact of noise.

In [ ]:
cudaq.set_target("nvidia")
print(cudaq.sample(test, noise_model = noise))


geometry = [('H', (0., 0., 0.)), ('H', (0., 0., .7474))]
molecule = solvers.create_molecule(geometry, 'sto-3g', 0, 0)

noise_empty = cudaq.NoiseModel()

numQubits = molecule.n_orbitals * 2
numElectrons = molecule.n_electrons
spin = 0
initialX = [-.2] * solvers.stateprep.get_num_uccsd_parameters(
    numElectrons, numQubits)

@cudaq.kernel
def uccsd():
    reg = cudaq.qvector(numQubits)
    for i in range(numElectrons):
        x(reg[i])
    solvers.stateprep.uccsd(reg, initialX, numElectrons, spin)

#noiseless value
print(cudaq.observe(uccsd, molecule.hamiltonian, noise_model = noise_empty).expectation())
noiseless = cudaq.observe(uccsd, molecule.hamiltonian, noise_model = noise_empty).expectation()

print(cudaq.draw(uccsd))

As a bonus, try repeating this exercise using a simulator tuned to mimic the noise of a physical QPU.  CUDA-Q provides access to, for example, the [Quantinuum H-2 emulator](https://nvidia.github.io/cuda-quantum/latest/using/backends/hardware/iontrap.html#quantinuum),  [IonQ's emulator](https://nvidia.github.io/cuda-quantum/latest/using/backends/hardware/iontrap.html#ionq), [Infleqtion's noisy simulator](https://nvidia.github.io/cuda-quantum/latest/using/backends/hardware/neutralatom.html#infleqtion), and more.

### Exercise 9 :
Now, write a function that computes the expectation values for various configurations of errors.  The function comments will guide you on the inputs and what the function should return.


In [ ]:
def get_noisy_data(e_type =[], gate=[] , qubit=[], prob=[], shots=-1, trajectories=None):
    """The function takes in various configurations of noise channels, builds a noise model, uses the noise model to obtain 40 expectation values,
       and the returns a list of the difference between the noisy expectation values and the noiseless.

    Parameters
    ----------
    e_type: list[int]
        List designating the type of each error applied for a given noise channel.
            1=phaseflip
            2=amplitude damping
            3=depolarization
    gate: list[str]
        List designating the type of gate each error is applied on (e.g. 'h' or'x').
    qubit: list[int]
        List designating the qubit index where the noise channel is applied.
    prob: list[float]
        List designates the probability of error for each noise channel
    shots: int
        Designates the number of shots used to compute the expectation value.  Default (-1) is exact, non-shot based result.
    trajectories: int
        Designates the number of trajecttores sampled when computing the expectation value.

    Returns
    -------
    list[float]
        List of length 40 where each elementis the diffence between the noisy and noiseless value.
    """

    #TODO

The function below, will take the result from `get_noisy_data` and plot them.  Just enter each output from `get_noisy_data` as an element of the `datasets` list variable, name the categories for the x-axis, and label the datasets if more than one is provided. The next cell provides an example.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_data(datasets, categories, labels=None):
    """
    Plots the mean and ±1 SD error bars for one or more datasets,
    with the y-axis centered at 0 for each category.

    Parameters:
    -----------
    datasets : list of lists (single dataset) or list of list of lists (multiple datasets)
        Each sub-list (or sub-sub-list if multiple) contains numeric values for a given category.
    categories : list of str
        Labels for the categories corresponding to each data sub-list.
    labels : list of str, optional
        Labels for each dataset if multiple datasets are provided.
    """
    # If only a single dataset (list of lists) is given, wrap it so we can loop uniformly
    if not isinstance(datasets[0][0], (list, np.ndarray)):
        datasets = [datasets]

    # If no labels given, auto-generate labels
    if labels is None:
        labels = [f"Dataset {i+1}" for i in range(len(datasets))]

    plt.figure()
    all_vals = []  # Collect all mean±std values to compute symmetric y-axis

    # Plot each dataset with a small horizontal offset
    for i, data in enumerate(datasets):
        means = [np.mean(vals) for vals in data]
        stds = [np.std(vals) for vals in data]
        x_positions = np.arange(len(categories)) + i * 0.1

        plt.errorbar(
            x_positions, means, yerr=stds, fmt='o', capsize=5,
            label=labels[i]
        )

        # Store values for y-axis range calculation
        all_vals.extend([m + s for m, s in zip(means, stds)])
        all_vals.extend([m - s for m, s in zip(means, stds)])

    # Center x-ticks between all plotted points
    midpoint_offset = ((len(datasets) - 1) * 0.1) / 2
    plt.xticks(np.arange(len(categories)) + midpoint_offset, categories)

    plt.ylabel('Mean Value (Expectation Value)')
    plt.title('Mean ±1 SD Error Bars (Centered at 0 = No Noise)')

    # Compute symmetric y-limits around 0
    y_upper = max(all_vals)
    y_lower = min(all_vals)
    max_range = max(abs(y_upper), abs(y_lower))
    plt.ylim([-max_range, max_range])

    plt.axhline(0, color='black', linewidth=0.8, linestyle='--')  # Reference line at y=0

    if len(datasets) > 1:
        plt.legend()

    plt.show()


#### Analyzing Shot Number ####

A source of noise not discussed thus far, and of a completely different nature than any physical noise channel, is sampling error. Sampling based quantum algorithms can produce inaccurate results simply due to sampling error, even if the hardware were perfect.  The code below, demonstrates how to use the `plot_data` function, and produces the distributions of hydrogen ground state energies obtained with anywhere from 10 to 10000 shots compared to 0 which is the noiseless result in the limit of infinite samples.

Notice how sampling based error is centered on the noiseless result and rapidly dissipates as more shots are used.   The rest of the simulations below will not perform shot based sampling so the only deviations from zero are due to the noise channels you implement below. Nevertheless, it is important to not forget that sampling error is a ubiquitous source of error for most quantum algorithms.

In [ ]:
data = [get_noisy_data(shots=10),
        get_noisy_data(shots=100),
        get_noisy_data(shots=1000),
        get_noisy_data(shots=10000),
]

categories = ['10','100', '1000', '10000' ]
labels =['series 1']

plot_data([data], categories, labels)



#### Analyzing Trajectory Number ####

Similar to sampling error, trajectory based noise simulation is also dependent on the number of trajectories used to compute the expectation value. Even if each trajectory is used to compute the expectation value exactly, if too few statevectors are sampled, the results of the noisy simulations can become unreliable.

Build a data set that only applies bitflip errors on $X$ gates for every qubit with probability 0.01. Vary the trajectories from 10 to 10000 and comment on the results.  

Does this noise model systematically over or under estimate the energy prediction? Could you be confident in this observation if you only used 10 trajectories?

In [ ]:
#TODO

#### Analyzing Error Probability and Gate Type ####

Now, turning to questions more specific to the structure of this algorithm's circuit, one can ask how changing the probability of a bitflip error impacts the computed energy. Plot the results corresponding to bitflip errors on all $X$ gates for all qubits with decreasing probabilities of .1, .01, .001, .0001. Plot a second series on the same graph but this time have the bitflip error applied on the $H$ gates.


For which type of gate are bitflip errors more problematic to the result?

In [ ]:
#TODO

#### Analyzing Error Type ####

Now, fix the error probability at 0.1 and this time vary the type of error.  Keep the two series with errors occurring on $X$ and $H$ gates.

Which type of error is most severe? Are there any errors that have little to no impact?

In [ ]:
#TODO

#### Analyzing Error Location (Qubit) ####

Finally, keep the two series and now perform only Amplitude Damping errors. This time, set all of the probabilities equal to 0, expect for the single error prone qubit which is set to a probability of 0.1.  

If you had a QPU where you knew qubit A was particularly prone to amplitude damping for some reason, when you compile the algorithm, which wire of the quantum circuit (q0, q1, q2, q3) should map to qubit A to ensure the best results based on your simulations?

In [ ]:
#TODO

### 3.3b: Zero Noise Extrapolation ###

QPU results today are sometimes improved using **quantum error mitigation (QEM)** techniques.  QEM techniques use classical postprocessing to improve results without the utilization of proper QEC protocols. One such QEM technique is **zero noise extrapolation (ZNE)**.  The idea behind ZNE is that it is really hard to remove noise from an algorithm run on a physical QPU, but it is very easy to add noise.

The ZNE process works by applying increasing factors of error through clever application of the identity operator. For example, consider a circuit composed of a single layer of $R_X$ rotations of $\pi$ radians. Applying the same gate three times is mathematically the same as applying it once and should have no impact on the result.

$$R_X(\pi)R_X(\pi)R_X(\pi) = R_X(\pi)I = R_X(\pi)$$

Experimentally, this is truly the identity operation as each gate is a noise channel and the total noise factor is increased from 1x to 3x.  If this procedure is continued (5x, 7x, 9x, ...) the data can be fit to a curve and extrapolated back to estimate the experimentally inaccessible case of a 0x noise factor!  So, paradoxically, adding noise can improve the result.

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-6-images/zneplot.png?raw=1" alt="Drawing" style="width: 1000px;"/>


ZNE is a useful technique because it can be used experimentally. Noisy circuit simulation can demonstrate its effectiveness and help benchmark the effectiveness of ZNE when used on a physical QPU, help refine noise models, and test other QEM techniques before running experiments.  




In [ ]:
### Exercise 10 :
You will now code an ZNE example by following the steps below:

1. Create a Random Hamiltonian for a larger (20 qubit circuit)
2. Define a quantum circuit with a layer of $R_x(\pi/2)$ gates followed by a layer of $X$ gates.
3. Put a bitflip error on the $X$ gates and an Amplitudes Damping error on the $R_X$ gates.
4. Perform ZNE to obtain a correction for each. (Hint: use the $\texttt{np.poly1d()}$ to fit a polynomial.)
5. Apply the correction to the original noisy circuit and calculate the percent error of the noisy circuit and the ZNE corrected result relative to the noiseless case.


In [ ]:
#Make Hamiltonian
#TODO

#Simulate noise and fit extrapolations
#TODO


def plot_zero_noise_extrapolation(noise_factors, measurements, poly_fit):
    """
    Plot the original data vs. noise factor and the polynomial fit extended
    down to noise=0 to show the extrapolation result.
    """
    # Create a range of noise values from 0 to slightly beyond the largest noise factor
    x_range = np.linspace(0, max(noise_factors) + 0.5, 50)
    y_fit = poly_fit(x_range)

    # Plot measured data points
    plt.scatter(noise_factors, measurements, label='Measured Data', color='blue')
    # Plot polynomial fit
    plt.plot(x_range, y_fit, label='Fit (degree = {})'.format(poly_fit.order), color='red')

    # Highlight the zero-noise extrapolation point
    extrapolated_value = poly_fit(0)
    plt.scatter([0], [extrapolated_value], color='green', zorder=5,
                label='Zero-Noise Extrapolation = {:.3f}'.format(extrapolated_value))

    plt.xlabel('Noise Factor')
    plt.ylabel('Measured Expectation Value')
    plt.title('Zero-Noise Extrapolation')
    plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    plt.legend()
    plt.grid(True)
    plt.show()
    print(f"Percent Error of ZNE Estimate {(extrapolated_value - noiseless)/noiseless*100} %")


print(f"Percent Error of Uncorrected Noisy Circuit: {(results[0] - noiseless)/noiseless*100} %")


plot_zero_noise_extrapolation(factors, results, linear)
plot_zero_noise_extrapolation(factors, results, quadratic)


### 3.3c: QEC Experiments ###



Noisy circuit simulation is perhaps most useful as a tools for QEC researchers.  One can test how a code will perform in a variety of different noise conditions.  Assuming an accurate noise model, this can be a great way to assess characteristics of new codes. Below you will add noise to the Steane code you prepared in lab 2.



In [ ]:

### Exercise 6 :
Apply noise to the Steane code in the following three ways and determine which case produces the best and worst logical error rates, keeping the probability of error fixed at 0.05.  In which cases is the logical error rate an improvement over the 0.05 error rate?
1. Use $\texttt{cudaq.apply\_noise(cudaq.XError, p, data\_qubits[j])}$ to manually apply Kraus operators following encoding of the Steane code but before the stabilizer checks are run.  These errors are not tied to gates but model errors induced while the system idles.
2. Now, use $\texttt{cudaq.apply\_noise(cudaq.Depolarization2, p, data\_qubits[i], data\_qubits[j])}$ to apply a depolarization error following all of the two qubit gates in the encoding circuit, whee q and r are the two qubits involved in the gate operation.
3. Apply a bitflip noise channel to all $\texttt{mz}$ measurements.  In this case, errors are also possible in measurements performed on the ancillas.  This helps model situations where measurements are performed in a way that is not fault tolerant.


In [ ]:
import cudaq

cudaq.set_target('stim')

p = 0.05
cudaq.unset_noise()
noise = cudaq.NoiseModel()

@cudaq.kernel
def steane_code():
    """Prepares a kernel for the Steane Code
    Returns
    -------
    cudaq.kernel
        Kernel for running the Steane code
    """
    data_qubits = cudaq.qvector(7)
    ancilla_qubits = cudaq.qvector(3)

    # Create a superposition over all possible combinations of parity check bits
    h(data_qubits[4])
    h(data_qubits[5])
    h(data_qubits[6])

    #Entangle states to enforce constraints of parity check matrix

    x.ctrl(data_qubits[0],data_qubits[1])
    x.ctrl(data_qubits[0],data_qubits[2])
    x.ctrl(data_qubits[4],data_qubits[0])
    x.ctrl(data_qubits[4],data_qubits[1])
    x.ctrl(data_qubits[4],data_qubits[3])

    x.ctrl(data_qubits[5],data_qubits[0])
    x.ctrl(data_qubits[5],data_qubits[2])
    x.ctrl(data_qubits[5],data_qubits[3])

    x.ctrl(data_qubits[6],data_qubits[1])
    x.ctrl(data_qubits[6],data_qubits[2])
    x.ctrl(data_qubits[6],data_qubits[3])

    # Detect Z errors
    h(ancilla_qubits)

    x.ctrl(ancilla_qubits[0],data_qubits[0])
    x.ctrl(ancilla_qubits[0],data_qubits[1])
    x.ctrl(ancilla_qubits[0],data_qubits[3])
    x.ctrl(ancilla_qubits[0],data_qubits[4])

    x.ctrl(ancilla_qubits[1],data_qubits[0])
    x.ctrl(ancilla_qubits[1],data_qubits[2])
    x.ctrl(ancilla_qubits[1],data_qubits[3])
    x.ctrl(ancilla_qubits[1],data_qubits[5])

    x.ctrl(ancilla_qubits[2],data_qubits[1])
    x.ctrl(ancilla_qubits[2],data_qubits[2])
    x.ctrl(ancilla_qubits[2],data_qubits[3])
    x.ctrl(ancilla_qubits[2],data_qubits[6])

    h(ancilla_qubits)

    sz1 = mz(ancilla_qubits[0])
    sz2 = mz(ancilla_qubits[1])
    sz3 = mz(ancilla_qubits[2])

    #Reset ancillas
    reset(ancilla_qubits)

    # Detect X errors
    h(ancilla_qubits)

    z.ctrl(ancilla_qubits[0],data_qubits[0])
    z.ctrl(ancilla_qubits[0],data_qubits[1])
    z.ctrl(ancilla_qubits[0],data_qubits[3])
    z.ctrl(ancilla_qubits[0],data_qubits[4])

    z.ctrl(ancilla_qubits[1],data_qubits[0])
    z.ctrl(ancilla_qubits[1],data_qubits[2])
    z.ctrl(ancilla_qubits[1],data_qubits[3])
    z.ctrl(ancilla_qubits[1],data_qubits[5])

    z.ctrl(ancilla_qubits[2],data_qubits[1])
    z.ctrl(ancilla_qubits[2],data_qubits[2])
    z.ctrl(ancilla_qubits[2],data_qubits[3])
    z.ctrl(ancilla_qubits[2],data_qubits[6])

    h(ancilla_qubits)

    sx1 = mz(ancilla_qubits[0])
    sx2 = mz(ancilla_qubits[1])
    sx3 = mz(ancilla_qubits[2])


    # Correct X errors

    if sx1 and sx2 and sx3:
        x(data_qubits[3])
    elif sx1 and sx2:
        x(data_qubits[0])
    elif sx1 and sx3:
        x(data_qubits[1])
    elif sx2 and sx3:
        x(data_qubits[2])
    elif sx1:
        x(data_qubits[4])
    elif sx2:
        x(data_qubits[5])
    elif sx3:
        x(data_qubits[6])



    # Correct Z errors

    if sz1 and sz2 and sz3:
        z(data_qubits[3])
    elif sz1 and sz2:
        z(data_qubits[0])
    elif sz1 and sz3:
        z(data_qubits[1])
    elif sz2 and sz3:
        z(data_qubits[2])
    elif sz1:
        z(data_qubits[4])
    elif sz2:
        z(data_qubits[5])
    elif sz3:
        z(data_qubits[6])

    mz(data_qubits)



results = cudaq.sample(steane_code, shots_count=10000, noise_model=noise)

print(results)

ones = 0
zeros = 0
for bitstring in results:
    counts = results.count(bitstring)
    parity = sum(int(bit) for bit in bitstring) % 2

    if parity == 0:
        zeros += 1*counts
    else:
        ones += 1*counts

logical_rate = ones/(zeros+ones)

print(f"logical error rate:{logical_rate}")


## 3.4: Using Dynamical Simulations to Build a Noise Model ###

The noise models used thus far are meant to mimic the underlying physics of physical qubits. Often, noise models are heavily informed by experiment, but extracting meaningful insights can be extremely difficult for such complex systems.

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-6-images/dynamics_noise_model.png?raw=1" alt="Drawing" style="width: 1200px;"/>

To help with this task, the physics of the qubits can also be simulated to better understand noise sources and improve interpretation of experimental data.  This sort of simulation is known as dynamical simulation and models the evolution of a quantum system over time as the system interacts with its environment.

Exercise:

The code below will help you walk through an example of using dynamical simulation to produce a noise model for a single qubit amplitude damping channel. Recall, the corresponding noise channel looks like this.

$$ \epsilon(\rho)  = \sqrt{1-p}*\rho + \sqrt{p}*\rho*0.5*(X+iY) $$

Thus, the goal is to simulate a simple qubit system to determine what $p$, the probability of energy loss resulting in decay to the ground state, is.

Dynamical simulation is its own topic that warrants a detailed introduction that will not be provided here.  Instead, the steps of the dynamical simulation will be discussed at a high level while curious readers can explore the CUDA-Q dynamics page for more information and more detailed examples.

To get started, import the following functions and libraries. This example will use the CUDA-Q dynamics backend, set like any other backend.

In [ ]:
import cudaq
from cudaq import spin, operators, ScalarOperator, Schedule, ScipyZvodeIntegrator
import numpy as np
import cupy as cp
import os
import matplotlib.pyplot as plt
cudaq.set_target("dynamics")

You will simulate a superconducting transmon qubit which is driven close to resonance, to produce so called Rabi oscillations. You will model a qubit which has a set resonant frequency, add a driving term that depends on time and will drive the system close to its resonant frequency, and by doing so, introduce a change in population from the $\ket{0}$ state to the  $\ket{1}$ state.  In other words, you will simulate thr underlying physics required to perform an $X$ gate.

The first step is to construct the Hamiltonian of the system. The Hamiltonian consists of a $Z$ term that encodes the resonant frequency for the qubit corresponding to the transition from the $\ket{0}$ to the $\ket{1}$ state. The second, is a driving term that applies a time dependent function  towards the qubits resonant frequency.  When this happens, Rabi oscillations occur and the population of the system changes from 100\% $\ket{0}$ to 100\% $\ket{1}$, that is, an $X$ gate applied to the qubit!






In [ ]:
### Exercise 11 :

The code below sets up the the problem Hamiltonian, defines the dimensions of the system and specifies the initial ground state. The terms have more meaning than described above, but their details are not relevant for the purposes of this exercise.


In [ ]:
omega_z = 10.0 * 2 * np.pi
omega_x = 2 * np.pi
omega_drive = 1 * omega_z

hamiltonian = 0.5 * omega_z * spin.z(0)
hamiltonian += omega_x * ScalarOperator(lambda t: np.cos(omega_drive * t)) * spin.x(0)

dimensions = {0: 2}

rho0 = cudaq.State.from_data(
    cp.array([[1.0, 0.0], [0.0, 0.0]], dtype=cp.complex128))

Dynamics simulations are performed numerically and require a time step specifying how the evolution operator is applied. For this problem it is setup below.

In [ ]:
t_final = np.pi / omega_x
dt = 2.0 * np.pi / omega_drive / 100
n_steps = int(np.ceil(t_final / dt)) + 1
steps = np.linspace(0, t_final, n_steps)
schedule = Schedule(steps, ["t"])

You are now ready to run the simulation using CUDA-Q's `evolve` function, which does require GPU access. The code cell below shows the baseline case where the qubit does not interact with its environment (i.e. there is no decoherence), and the plot shows a the probability of sampling the $\ket{1}$ state rise from 0 to 1.  This corresponds to a pulse used to implement a perfect noiseless $X$ gate.

In [ ]:
evolution_result = cudaq.evolve(hamiltonian,
                                dimensions,
                                schedule,
                                rho0,
                                observables=[operators.number(0)],
                                collapse_operators=[],
                                store_intermediate_results=True,
                                integrator=ScipyZvodeIntegrator())

get_result = lambda idx, res: [
    exp_vals[idx].expectation() for exp_vals in res.expectation_values()
]
ideal_results = [
    get_result(0, evolution_result)]



plt.figure()
plt.plot(steps, ideal_results[0])
plt.ylabel("Probability(1)")
plt.xlabel("Time")
plt.title("No Decoherence")

Now, things can get a bit more interesting when decoherence is factored in.  In this case, a so called collapse operator is added to the dynamical simulation to model the decay of energy into the environment as the system evolves. In this case the simple collapse operator `np.sqrt(gamma_sm) * spin.minus(0)` is added.  `gamma_sm` is set to 1, but would in practice be determined by some experimental quantity.  Now, by running this simulation, you will be able to simulate amplitude damping and see what the probability of the $\ket{1}$ state remains after the pulse.

In [ ]:
gamma_sm = 1.0
evolution_result_decay = cudaq.evolve(
    hamiltonian,
    dimensions,
    schedule,
    rho0,
    observables=[operators.number(0)],
    collapse_operators=[
        np.sqrt(gamma_sm) * spin.minus(0),
    ],
    store_intermediate_results=True,
    integrator=ScipyZvodeIntegrator())

decoherence_results = [
    get_result(0, evolution_result_decay)
]

plt.figure()
plt.plot(steps, decoherence_results[0])
plt.ylabel("Probability(1)")
plt.xlabel("Time")
plt.title("Decoherence")

The pulse now has a peak of around .85, meaning $p=.15$ is a reasonable choice to parameterize an amplitude damping channel.

In [ ]:
cudaq.set_target('density-matrix-cpu')
cudaq.set_random_seed(13)
noise = cudaq.NoiseModel()

amplitude_damping = cudaq.AmplitudeDampingChannel(0.15) # this value is from the dynamics simulation
noise.add_channel('x', [0], amplitude_damping)

kernel = cudaq.make_kernel()
qubit = kernel.qalloc()
kernel.x(qubit)
kernel.mz(qubit)
counts = cudaq.sample(kernel, noise_model=noise)
prob_1 = counts.probability("1")
print("Probability of |1> from the gate-level simulation with noise:", prob_1)

cudaq.reset_target()

This is a very simple example, and in practice it is much harder to derive noise models from dynamical simulations.  Nevertheless, they are powerful tools for understanding noise.  You can also tweak other aspects of the simulation for more complex situations.  For example, try the following and see how they might change the amplitude damping parameterization.

1. change `omega_drive = 0.95 * omega_z` to be close to but not the same as the resonance frequency.
2. Add 0.1 to `t_final = np.pi / omega_x` to test what would happen if a gate pulse is applied for too long.

In [ ]:
# TO DO